# **Part1. 학습 데이터셋**

## 1. **Setting**

In [1]:
# 패키지 버전 고정 — 최초 1회 실행 후 커널 재시작 권장
%pip install -q "numpy==2.2.6" "pandas==2.2.3" "openai==1.109.1" "tqdm==4.67.1" "nbformat>=5.10,<6" "datasets" "huggingface_hub"

Note: you may need to restart the kernel to use updated packages.


In [46]:
# 라이브러리 import 및 버전 점검 — import는 Setting에 집중함
import os
import re
import json
import random
import inspect
from pathlib import Path
from datetime import datetime
from collections import Counter
from importlib.metadata import version
from huggingface_hub import login
import datasets
import openai

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

# OpenAI API key: 환경변수 또는 실행 시 입력 (노트북에 저장되지 않음)
from getpass import getpass
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key (not saved in notebook): ").strip()
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is required")

# Hugging Face 로그인 (새 토큰은 HF_TOKEN 환경변수 또는 실행 중 보안 입력창에서 받음)
# 주의: Hugging Face 토큰을 노트북 코드에 직접 붙여넣거나 출력하지 마세요
from getpass import getpass
from huggingface_hub import login, HfApi
HF_TOKEN = (os.getenv("HF_TOKEN") or "").strip()
if not HF_TOKEN:
    HF_TOKEN = getpass("새 Hugging Face access token 입력 (입력값 숨김): ").strip()
if not HF_TOKEN:
    raise ValueError("HF_TOKEN이 필요합니다. 새 토큰을 입력하거나 환경변수 HF_TOKEN을 설정하세요.")
login(token=HF_TOKEN)
HF_USERNAME = (os.getenv("PILATES_HF_USERNAME") or HfApi().whoami(token=HF_TOKEN)["name"]).strip()
print("Hugging Face 로그인 확인:", HF_USERNAME)  # 토큰은 출력하지 않음


EXPECTED = {"numpy":"2.2.6", "pandas":"2.2.3", "openai":"1.109.1", "tqdm":"4.67.1"}
for lib, expected in EXPECTED.items():
    actual = version(lib)
    print(f"{lib}: {actual} (권장 {expected})")
    if actual != expected:
        print(f"[주의] {lib} 버전 불일치. Setting 설치 후 커널 재시작 권장")

REFERENCE_DATE = "2026-10-30 13:00"
CURRENT_USER_ID = "U001"
RNG = random.Random(42)
MODEL = "gpt-4o"
CHAT_DATE = "2026-11-02"
NUM_SAMPLES = 80
SAVE_EVERY = 1  # PASS 1건마다 체크포인트 저장; 중단 시 손실 최소화
OUTPUT_DIR = Path("outputs_v2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_DIR / "generation_checkpoint_5to6_close_120.json"  # 이전 버전 데이터와 혼합 방지
HF_REPO_ID = os.getenv("PILATES_DATASET_REPO", f"{HF_USERNAME}/pilates-function-calling-datasets-korean-v2")

numpy: 2.2.6 (권장 2.2.6)
pandas: 2.2.3 (권장 2.2.3)
openai: 1.109.1 (권장 1.109.1)
tqdm: 4.67.1 (권장 4.67.1)


새 Hugging Face access token 입력 (입력값 숨김):  ········


Hugging Face 로그인 확인: somsom317


In [3]:
import torch

print("GPU 사용 가능 여부:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("사용 중인 GPU 장치명:", torch.cuda.get_device_name(0))
    print("할당된 GPU 개수:", torch.cuda.device_count())

GPU 사용 가능 여부: True
사용 중인 GPU 장치명: NVIDIA A100-SXM4-80GB
할당된 GPU 개수: 1


## **2. Data Load**

In [31]:
# DB 로딩
cwd = Path.cwd()
candidates = [cwd / "database", cwd / "Part6" / "database", cwd, Path("/mnt/data")]
DATA_DIR = next((p for p in candidates if (p / "users(1).json").is_file() or (p / "users.json").is_file()), None)
if DATA_DIR is None:
    raise FileNotFoundError("database 폴더에 users.json 등 8개 JSON을 넣어주세요.")

def load_json(filename, alternative=None):
    path = DATA_DIR / filename
    if not path.exists() and alternative:
        path = DATA_DIR / alternative
    with open(path, encoding="utf-8") as f:
        return pd.DataFrame(json.load(f))

df_users = load_json("users(1).json", "users.json")
df_memberships = load_json("memberships(2).json", "memberships.json")
df_classes = load_json("classes(3).json", "classes.json")
df_instructors = load_json("instructors(3).json", "instructors.json")
df_user_memberships = load_json("user_memberships(4).json", "user_memberships.json")
df_bookings = load_json("bookings(4).json", "bookings.json")
df_attendance_logs = load_json("attendance_logs(1).json", "attendance_logs.json")
df_policies = load_json("policies(2).json", "policies.json")

required = {"classes": (df_classes, ["day_of_week"]),
            "bookings": (df_bookings, ["class_date", "week_status"])}
for name, (df, cols) in required.items():
    missing = [col for col in cols if col not in df.columns]
    if missing:
        raise ValueError(f"{name} 데이터가 이전 버전입니다: {missing}. 최신 JSON으로 교체해 주세요.")
print("DATA_DIR:", DATA_DIR)
for name, df in [("users",df_users),("memberships",df_memberships),
                 ("classes",df_classes),("instructors",df_instructors),
                 ("user_memberships",df_user_memberships),("bookings",df_bookings),
                 ("policies",df_policies)]:
    print(f"{name:18s}: {len(df)} rows")

DATA_DIR: /workspace/database
users             : 50 rows
memberships       : 17 rows
classes           : 100 rows
instructors       : 8 rows
user_memberships  : 81 rows
bookings          : 105 rows
policies          : 15 rows


## **3. Functions**

In [32]:
# 2-1. 이용권 상품 정보 조회 함수
def get_membership_info(
    sessions=None,
    product_type=None,
    lesson_type=None
):
    membership_info = df_memberships.copy()

    # 전체 횟수 기준 상품 조회
    if sessions is not None:
        membership_info = membership_info[
            (membership_info['total_sessions'] == sessions)
        ]

    # 상품 유형 검색
    if product_type is not None:
        membership_info = membership_info[
            membership_info['product_type'] == product_type
        ]

    # 레슨 유형 검색
    if lesson_type is not None:
        membership_info = membership_info[
            membership_info['lesson_type'] == lesson_type
        ]

    if membership_info.empty:
        return {
            "success": False,
            "membership_count": 0,
            "message": "조건에 해당하는 이용권이 없습니다."
        }

    return {
        "success": True,
        "membership_count": len(membership_info),
        "membership_info": membership_info.to_dict(orient="records")
    }

In [33]:
# 2-2. 수업 시간표 조회 함수
def get_class_schedule(
    day_of_week=None,
    lesson_type=None,
    instructor_name=None,
    start_time=None,
    end_time=None,
    time_filter="exact"
):
    # 수업 정보와 강사 정보 병합
    class_info = df_classes.merge(
        df_instructors[['instructor_id', 'name']],
        on='instructor_id',
        how='left'
    ).rename(columns={'name': 'instructor_name'})

    # 요일별 조회
    if day_of_week is not None:
        class_info = class_info[
            class_info['day_of_week'] == day_of_week
        ]

    # 레슨 유형별 조회
    if lesson_type is not None:
        class_info = class_info[
            class_info['lesson_type'] == lesson_type
        ]

    # 강사 이름별 조회
    if instructor_name is not None:
        class_info = class_info[
            class_info['instructor_name'] == instructor_name
        ]

    # 시간 검색 방식 검증 (초과/미만과 이상/이하 구분)
    if time_filter not in ("exact", "after", "at_or_after", "before", "at_or_before", "between"):
        return {
            "success": False,
            "class_count": 0,
            "message": "지원하지 않는 시간 검색 방식입니다."
        }

    if time_filter == "between":
        # 시작 시각이 두 경계 사이인 수업 조회 (양 끝 포함)
        if start_time is None or end_time is None:
            return {
                "success": False,
                "class_count": 0,
                "message": "시간 범위 조회에는 시작 시간과 종료 시간이 모두 필요합니다."
            }

        class_info = class_info[
            (class_info['start_time'] >= start_time) &
            (class_info['start_time'] <= end_time)
        ]

    else:
        # 시작 시각 비교: after/before는 경계 제외
        if start_time is not None:
            if time_filter == "after":
                class_info = class_info[
                    class_info['start_time'] > start_time
                ]
            elif time_filter == "at_or_after":
                class_info = class_info[
                    class_info['start_time'] >= start_time
                ]
            elif time_filter == "before":
                class_info = class_info[
                    class_info['start_time'] < start_time
                ]
            elif time_filter == "at_or_before":
                class_info = class_info[
                    class_info['start_time'] <= start_time
                ]
            else:
                class_info = class_info[
                    class_info['start_time'] == start_time
                ]

        # between 외에는 end_time 이하에 종료되는 수업 조회
        if end_time is not None:
            class_info = class_info[
                class_info['end_time'] <= end_time
            ]

    # 조회 결과가 없는 경우
    if class_info.empty:
        return {
            "success": False,
            "class_count": 0,
            "message": "조건에 해당하는 수업 시간표가 없습니다."
        }

    # 요일 및 시간순 정렬
    day_order = {
        '월요일': 0,
        '화요일': 1,
        '수요일': 2,
        '목요일': 3,
        '금요일': 4
    }

    class_info = class_info.copy()
    class_info['_day_order'] = class_info['day_of_week'].map(day_order)

    class_info = class_info.sort_values(
        ['_day_order', 'start_time', 'instructor_name']
    )

    # 반환할 수업 정보
    columns = [
        'class_id',
        'day_of_week',
        'start_time',
        'end_time',
        'lesson_type',
        'instructor_id',
        'instructor_name',
        'capacity',
        'status'
    ]

    return {
        "success": True,
        "class_count": len(class_info),
        "class_schedule": class_info[columns].to_dict(orient="records")
    }

In [ ]:
# 2-3. 예약 가능한 수업 조회 함수
def get_available_classes(
    day_of_week=None,
    lesson_type=None,
    instructor_name=None,
    start_time=None,
    end_time=None,
    time_filter="exact"
):
    # 1. 개발 환경 기준 시점
    now = pd.Timestamp(REFERENCE_DATE)

    # 예약 오픈 기준: 매주 금요일 12시
    first_open = pd.Timestamp("2026-10-30 12:00")

    if now < first_open:
        return {
            "success": False,
            "available_class_count": 0,
            "message": "아직 예약이 오픈되지 않았습니다."
        }

    days_since_friday = (now.weekday() - 4) % 7
    open_date = now.normalize() - pd.Timedelta(
        days=days_since_friday
    )

    if now < open_date + pd.Timedelta(hours=12):
        open_date -= pd.Timedelta(days=7)

    # 예약이 열린 다음 주 월~금 날짜 생성
    monday = open_date + pd.Timedelta(days=3)

    target_dates = pd.DataFrame({
        "class_date": [
            (monday + pd.Timedelta(days=i)).strftime("%Y-%m-%d")
            for i in range(5)
        ],
        "day_of_week": [
            "월요일", "화요일", "수요일", "목요일", "금요일"
        ]
    })

    # 2. 수업과 강사 정보 병합
    class_info = df_classes.merge(
        df_instructors[["instructor_id", "name"]],
        on="instructor_id",
        how="left"
    ).rename(columns={"name": "instructor_name"})

    # 운영 중인 수업만 조회
    class_info = class_info[
        class_info["status"] == "ACTIVE"
    ].copy()

    # 정규 시간표와 예약 대상 날짜 연결
    class_info = class_info.merge(
        target_dates,
        on="day_of_week",
        how="inner"
    )

    # 3. 검색 조건 적용
    if day_of_week is not None:
        class_info = class_info[
            class_info["day_of_week"] == day_of_week
        ]

    if lesson_type is not None:
        class_info = class_info[
            class_info["lesson_type"] == lesson_type
        ]

    if instructor_name is not None:
        class_info = class_info[
            class_info["instructor_name"] == instructor_name
        ]

    # 시간 검색 방식 검증 (초과/미만과 이상/이하 구분)
    if time_filter not in ("exact", "after", "at_or_after", "before", "at_or_before", "between"):
        return {
            "success": False,
            "available_class_count": 0,
            "message": "지원하지 않는 시간 검색 방식입니다."
        }

    # 4. 시작 시각 조건 적용 (between은 양 끝 포함)
    if time_filter == "between":
        if start_time is None or end_time is None:
            return {
                "success": False,
                "available_class_count": 0,
                "message": "시간 범위 조회에는 두 시간이 필요합니다."
            }

        class_info = class_info[
            (class_info["start_time"] >= start_time) &
            (class_info["start_time"] <= end_time)
        ]

    else:
        if start_time is not None:
            if time_filter == "after":
                class_info = class_info[
                    class_info["start_time"] > start_time
                ]
            elif time_filter == "at_or_after":
                class_info = class_info[
                    class_info["start_time"] >= start_time
                ]
            elif time_filter == "before":
                class_info = class_info[
                    class_info["start_time"] < start_time
                ]
            elif time_filter == "at_or_before":
                class_info = class_info[
                    class_info["start_time"] <= start_time
                ]
            else:
                class_info = class_info[
                    class_info["start_time"] == start_time
                ]

        if end_time is not None:
            class_info = class_info[
                class_info["end_time"] <= end_time
            ]

    # 5. 해당 주차의 활성 예약 내역
    active_bookings = df_bookings[
        df_bookings["week_status"] == "ACTIVE"
    ].copy()

    active_bookings["class_date"] = (
        pd.to_datetime(active_bookings["class_date"])
        .dt.strftime("%Y-%m-%d")
    )

    active_bookings = active_bookings[
        active_bookings["class_date"].isin(
            target_dates["class_date"]
        )
    ]

    # 6. 확정 예약 인원 계산
    confirmed = active_bookings[
        active_bookings["status"] == "CONFIRMED"
    ]

    booking_counts = (
        confirmed.groupby(["class_id", "class_date"])
        .size()
        .reset_index(name="booked_count")
    )

    # 예약이 없는 수업도 포함하도록 LEFT JOIN
    class_info = class_info.merge(
        booking_counts,
        on=["class_id", "class_date"],
        how="left"
    )

    class_info["booked_count"] = (
        class_info["booked_count"].fillna(0).astype(int)
    )

    # 7. 잔여석 계산
    class_info["remaining_seats"] = (
        class_info["capacity"] - class_info["booked_count"]
    ).clip(lower=0)

    # 잔여석이 있는 수업만 반환
    class_info = class_info[
        class_info["remaining_seats"] > 0
    ].copy()

    # 8. 날짜 및 시간순 정렬
    class_info = class_info.sort_values(
        ["class_date", "start_time", "instructor_name"]
    )

    columns = [
        "class_id",
        "class_date",
        "day_of_week",
        "start_time",
        "end_time",
        "lesson_type",
        "instructor_name",
        "capacity",
        "booked_count",
        "remaining_seats"
    ]

    return {
        "success": True,
        "week_start_date": monday.strftime("%Y-%m-%d"),
        "available_class_count": len(class_info),
        "available_classes": class_info[columns].to_dict(
            orient="records"
        ),
        "message": (
            "예약 가능한 수업 조회가 완료되었습니다."
            if not class_info.empty
            else "조건에 해당하는 예약 가능한 수업이 없습니다."
        )
    }

In [35]:
# 2-4. 내 보유 이용권 조회 함수
# 실제 서비스에서는 CURRENT_USER_ID 대신 인증된 세션의 user_id 사용

def get_my_memberships():
    user_id = CURRENT_USER_ID
    user = df_users.loc[df_users["user_id"] == user_id, ["user_id", "name", "status"]]
    if user.empty:
        return {"success": False, "membership_count": 0, "message": "로그인 회원 정보를 찾을 수 없습니다."}

    info = df_user_memberships.loc[df_user_memberships["user_id"] == user_id].merge(
        df_memberships[["membership_id", "membership_name", "product_type", "lesson_type", "components", "total_sessions"]],
        on="membership_id", how="left"
    ).copy()
    today = pd.Timestamp(REFERENCE_DATE).normalize()
    info["is_usable"] = (
        info["status"].eq("ACTIVE") &
        pd.to_datetime(info["start_date"], errors="coerce").le(today) &
        pd.to_datetime(info["expires_at"], errors="coerce").ge(today) &
        pd.to_numeric(info["remaining_sessions"], errors="coerce").gt(0)
    )
    columns = ["user_membership_id", "membership_id", "membership_name",
               "product_type", "lesson_type", "components", "total_sessions",
               "start_date", "expires_at", "remaining_sessions",
               "remaining_sessions_by_type", "status", "is_usable"]
    records = info[columns].astype(object).where(pd.notna(info[columns]), None).to_dict("records")
    return {
        "success": True,
        "user_id": user_id,
        "user_name": user.iloc[0]["name"],
        "reference_date": today.strftime("%Y-%m-%d"),
        "membership_count": len(records),
        "user_memberships": records,
        "message": "내 이용권 조회 완료" if records else "보유 이용권이 없습니다."
    }

In [36]:
# 2-5. 내 예약 내역 조회 함수
def get_my_bookings(status=None, week_status=None, day_of_week=None):
    user_id = CURRENT_USER_ID
    user = df_users.loc[df_users["user_id"] == user_id, ["user_id", "name"]]
    if user.empty:
        return {"success": False, "booking_count": 0,
                "message": "로그인 회원 정보를 찾을 수 없습니다."}

    info = df_bookings.loc[df_bookings["user_id"] == user_id].copy()
    info = info.merge(
        df_classes[["class_id", "day_of_week", "start_time", "end_time",
                    "lesson_type", "instructor_id"]],
        on="class_id", how="left"
    ).merge(
        df_instructors[["instructor_id", "name"]].rename(columns={"name": "instructor_name"}),
        on="instructor_id", how="left"
    )
    if status is not None:
        info = info[info["status"] == status]
    if week_status is not None:
        info = info[info["week_status"] == week_status]
    if day_of_week is not None:
        info = info[info["day_of_week"] == day_of_week]

    info = info.sort_values(["class_date", "start_time", "booked_at"], ascending=[False, True, False])
    columns = ["booking_id", "class_id", "class_date", "day_of_week",
               "start_time", "end_time", "lesson_type", "instructor_name",
               "booked_at", "status", "week_status", "waitlist_position",
               "cancelled_at", "user_membership_id", "credit_deducted", "credit_restored"]
    records = info[columns].astype(object).where(pd.notna(info[columns]), None).to_dict("records")
    return {"success": True, "user_id": user_id,
            "user_name": user.iloc[0]["name"], "booking_count": len(records),
            "bookings": records,
            "message": "내 예약 내역 조회 완료" if records else "조건에 맞는 예약 내역이 없습니다."}

In [37]:
# 2-6. 약관 및 운영 정책 조회 함수

def search_policy_info(keyword=None):
    info = df_policies.copy()
    if keyword is not None:
        keyword = keyword.strip()
        if keyword:
            info = info[
                info["keyword"].astype(str).str.contains(keyword, case=False, regex=False, na=False) |
                info["content"].astype(str).str.contains(keyword, case=False, regex=False, na=False)
            ]
    if info.empty:
        return {"success": False, "policy_count": 0, "policies": [],
                "message": "조건에 해당하는 약관이 없습니다."}
    return {"success": True, "policy_count": len(info),
            "policies": info[["policy_id", "keyword", "content", "effective_date"]].to_dict("records")}

## **4. Tool Schema**

In [38]:
# 필라테스 Function Calling Tool 명세 (총 6개)
tools = [
    {
        "type": "function",
        "name": "get_membership_info",
        "description": (
            "센터에 등록된 이용권 상품의 가격, 전체 이용 횟수, 유효기간, 구성 정보를 조회합니다. "
            "전체 이용 횟수(total_sessions)는 sessions로 검색하며 SINGLE과 COMBO 모두 적용됩니다. "
            "SINGLE 상품은 lesson_type과 sessions로 검색할 수 있습니다. "
            "COMBO 상품은 product_type='COMBO'로 검색하고 반환된 components에서 구성별 레슨 유형과 횟수를 확인합니다. "
            "콤보의 모든 구성과 횟수가 명시되어 정확한 전체 횟수를 알 수 있다면 합계를 sessions로 전달합니다. "
            "일부 구성만 알려졌거나 특정 구성이 포함된 모든 상품을 찾을 때는 sessions를 생략합니다. "
            "본인의 보유 이용권과 잔여 횟수는 get_my_memberships로 확인합니다. "
            "조회 결과의 status를 확인하며, 반환된 모든 상품이 판매 중이라고 가정하지 않습니다."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "sessions": {
                    "type": "integer",
                    "description": (
                        "상품의 전체 이용 횟수(total_sessions)에 정확히 일치하는 정수입니다. "
                        "SINGLE과 COMBO 상품 모두에 적용됩니다. "
                        "'6회권 있어?'는 sessions=6으로 검색합니다. "
                        "'총 60회 콤보권 있어?'는 sessions=60, product_type='COMBO'로 검색합니다. "
                        "콤보 상품에서 전체 구성을 모두 알 수 있고 그 구성과 정확히 일치하는 상품을 찾는다면, "
                        "각 구성의 sessions를 합산하여 전달합니다. "
                        "예: '개인 6회 + 그룹 32회 콤보권'은 6+32=38이므로 sessions=38, product_type='COMBO'입니다. "
                        "예: '개인 12회 + 그룹 48회 콤보권'은 12+48=60이므로 sessions=60, product_type='COMBO'입니다. "
                        "단, total_sessions가 같아도 구성은 다를 수 있으므로 반환된 components를 반드시 검증합니다. "
                        "'개인 12회가 포함된 콤보권'처럼 일부 구성만 지정했거나 다른 구성이 추가될 수 있는 '포함' 검색이면 "
                        "전체 횟수를 확정할 수 없으므로 sessions를 생략하고 components로 판단합니다.(중요) "
                        "'몇 회짜리가 있어?'처럼 횟수가 없는 경우에도 생략합니다. "
                        "존재 여부가 불확실한 횟수도 임의로 수정하지 않고 그대로 전달합니다. "
                        "구성별 개별 횟수(예: 6 또는 32)를 전체 횟수인 sessions로 잘못 전달하지 않습니다."
                    )
                },
                "product_type": {
                    "type": "string",
                    "enum": ["SINGLE", "COMBO"],
                    "description": (
                        "SINGLE은 한 종류의 레슨으로 구성된 단일 이용권, "
                        "COMBO는 여러 레슨 유형을 결합한 이용권입니다. "
                        "'단일권'을 명시하면 SINGLE을 선택하고, "
                        "'개인+그룹', '듀엣+그룹', '복합', '혼합', '콤보'를 요청하면 COMBO를 선택합니다. "
                        "정확한 전체 횟수를 아는 COMBO 검색에서는 sessions를 함께 전달할 수 있습니다. "
                        "예: '개인 6회 + 그룹 32회 콤보권'은 product_type='COMBO', sessions=38입니다. "
                        "COMBO의 구성별 횟수는 반환된 components에서 비교합니다. "
                        "특정 구성을 '포함'하는 상품은 해당 구성이 존재하는지 확인하고, "
                        "'정확히 해당 구성'을 요구하면 추가 구성 없이 전체 구성이 일치하는지 확인합니다. "
                        "COMBO 검색에는 lesson_type을 함께 전달하지 않습니다. "
                        "상품 유형이 지정되지 않으면 product_type을 생략하여 둘 다 검색할 수 있습니다. "
                        "단, lesson_type을 전달하면 데이터 구조상 그 레슨 유형의 SINGLE 상품만 조회됩니다."
                    )
                },
                "lesson_type": {
                    "type": "string",
                    "enum": ["7:1 레슨", "2:1 듀엣 레슨", "1:1 개인 레슨"],
                    "description": (
                        "SINGLE 상품의 레슨 유형을 정확히 검색합니다. "
                        "'그룹', '그룹 수업', '그룹 레슨', '7:1'은 '7:1 레슨'으로, "
                        "'듀엣', '듀엣 수업', '듀엣 레슨', '2:1'은 '2:1 듀엣 레슨'으로, "
                        "'개인', '개인 수업', '개인 레슨', '1:1'은 '1:1 개인 레슨'으로 변환합니다. "
                        "'개인 12회 단일권'은 lesson_type='1:1 개인 레슨', sessions=12, product_type='SINGLE'로 검색합니다. "
                        "COMBO는 수업 종류가 아니라 상품 구성 방식이므로 lesson_type에 COMBO를 넣지 않습니다. "
                        "콤보 상품의 구성 레슨은 product_type='COMBO'로 조회한 뒤 components에서 비교합니다. "
                        "'개인 12회가 포함된 모든 상품'처럼 SINGLE과 COMBO를 모두 찾는 질문이면 "
                        "SINGLE은 lesson_type과 sessions로, COMBO는 product_type='COMBO'로 각각 조회하고 components를 확인합니다. "
                        "레슨 유형이 지정되지 않으면 생략합니다. "
                        "'7:1 듀엣'처럼 표현이 충돌하면 임의로 선택하지 않고 확인합니다."
                    )
                }
            },
            "required": [],
            "additionalProperties": False
        }
    },

 {'type': 'function',
  'name': 'get_class_schedule',
  'description': "월~금 정규 수업 시간표를 조회합니다. '수업 언제 있어?', '강사 시간표 알려줘'처럼 운영 일정을 물을 때 사용합니다. 시간표는 정기 편성 정보이며 예약 잔여석을 "
                 "확인하지 않습니다. '빈자리', '예약 가능'은 get_available_classes를 사용합니다. 월·수, 3시이후나 6시 이후처럼 여러 요일이나 요일별 다른 시간 조건은 각각 별도 "
                 '호출합니다.',
  'parameters': {'type': 'object',
                 'properties': {'day_of_week': {'type': 'string',
                                                'enum': ['월요일', '화요일', '수요일', '목요일', '금요일'],
                                                'description': "단일 요일만 입력합니다. '월·수'는 월요일, 수요일로 나누어 각각 호출합니다. 생략하면 "
                                                               '월~금 전체를 조회합니다.'},
                                'lesson_type': {'type': 'string',
                                                'enum': ['7:1 레슨', '2:1 듀엣 레슨', '1:1 개인 레슨'],
                                                'description': "실제로 편성된 수업의 레슨 유형입니다. '그룹', '그룹 수업', '그룹 레슨', "
                                                               "'7:1'은 '7:1 레슨', '듀엣', '듀엣 수업', '듀엣 레슨', '2:1'은 "
                                                               "'2:1 듀엣 레슨', '개인', '개인 수업', '개인 레슨', '1:1'은 '1:1 "
                                                               "개인 레슨'으로 해석합니다. 콤보는 실제 수업 유형이 아니라 이용권의 구성 방식이므로 "
                                                               '수업 lesson_type에 COMBO를 넣지 않습니다. 레슨 유형을 지정하지 않으면 '
                                                               "전체 유형을 검색합니다. 서로 충돌하는 유형 표현(예: '7:1 듀엣')이 있으면 임의로 "
                                                               '선택하지 않고 확인합니다.'},
                                'instructor_name': {'type': 'string',
                                                    'description': '강사명 정확 일치 검색입니다. 사용자가 지정한 강사 이름을 전달하고, 이름이 '
                                                                   '불확실하면 확인합니다. 생략하면 전체 강사입니다.'},
                                'start_time': {'type': 'string',
                                               'description': "수업 시작 시각, HH:MM 형식입니다(예: 오후 3시='15:00'). "
                                                              'time_filter와 함께 지정합니다.'},
                                'end_time': {'type': 'string',
                                             'description': 'HH:MM 형식입니다. between에서는 수업 시작 시각 검색 범위의 상한(양 끝 포함), '
                                                            '그 외에는 수업 자체 종료 시각의 상한(이하)입니다.'},
                                'time_filter': {'type': 'string',
                                                'enum': ['exact',
                                                         'after',
                                                         'at_or_after',
                                                         'before',
                                                         'at_or_before',
                                                         'between'],
                                                'description': "수업의 시작 시각 기준: exact=정각 시작(==), before='이전' 미만(<), "
                                                               "at_or_before='까지' 이하(<=), after='초과' 기준 시각 제외(>), "
                                                               "at_or_after='부터' 이상(>=), between='A부터 B까지' 시작 시각 "
                                                               '구간 양끝 포함입니다. 예: 3시 이전은 15:00 제외, 3시부터는 15:00 포함. '
                                                               '생략하면 exact입니다.'}},
                 'required': [],
                 'additionalProperties': False}},
 {'type': 'function',
  'name': 'get_available_classes',
  'description': "예약이 오픈된 주차에서 실제 잔여석이 1석 이상인 수업만 조회합니다. '빈자리', '자리 남았어?', '예약 가능한 시간'일 때 사용합니다. 수업 예약 현황을 집계하므로 "
                 "별도의 내 예약 조회 없이 일반 잔여석을 확인할 수 있지만, 회원의 이용권 사용 자격이나 같은 날짜 중복 예약 여부는 판단하지 않습니다. '내 이용권으로 김민지 선생님 수업 예약 "
                 "가능?'이라면 get_my_memberships 및 get_my_bookings 결과와 함께 확인합니다. 만석 수업과 대기 명단은 반환하지 않습니다. 월·수처럼 서로 다른 "
                 '요일 조건은 별도 호출합니다.',
  'parameters': {'type': 'object',
                 'properties': {'day_of_week': {'type': 'string',
                                                'enum': ['월요일', '화요일', '수요일', '목요일', '금요일'],
                                                'description': "단일 요일만 입력합니다. '월·수'는 월요일, 수요일로 나누어 각각 호출합니다. 생략하면 "
                                                               '월~금 전체를 조회합니다.'},
                                'lesson_type': {'type': 'string',
                                                'enum': ['7:1 레슨', '2:1 듀엣 레슨', '1:1 개인 레슨'],
                                                'description': "실제로 편성된 수업의 레슨 유형입니다. '그룹', '그룹 수업', '그룹 레슨', "
                                                               "'7:1'은 '7:1 레슨', '듀엣', '듀엣 수업', '듀엣 레슨', '2:1'은 "
                                                               "'2:1 듀엣 레슨', '개인', '개인 수업', '개인 레슨', '1:1'은 '1:1 "
                                                               "개인 레슨'으로 해석합니다. 콤보는 실제 수업 유형이 아니라 이용권의 구성 방식이므로 "
                                                               '수업 lesson_type에 COMBO를 넣지 않습니다. 레슨 유형을 지정하지 않으면 '
                                                               "전체 유형을 검색합니다. 서로 충돌하는 유형 표현(예: '7:1 듀엣')이 있으면 임의로 "
                                                               '선택하지 않고 확인합니다.'},
                                'instructor_name': {'type': 'string',
                                                    'description': '강사명 정확 일치 검색입니다. 사용자가 지정한 강사 이름을 전달하고, 이름이 '
                                                                   '불확실하면 확인합니다. 생략하면 전체 강사입니다.'},
                                'start_time': {'type': 'string',
                                               'description': "수업 시작 시각, HH:MM 형식입니다(예: 오후 3시='15:00'). "
                                                              'time_filter와 함께 지정합니다.'},
                                'end_time': {'type': 'string',
                                             'description': 'HH:MM 형식입니다. between에서는 수업 시작 시각 검색 범위의 상한(양 끝 포함), '
                                                            '그 외에는 수업 자체 종료 시각의 상한(이하)입니다.'},
                                'time_filter': {'type': 'string',
                                                'enum': ['exact',
                                                         'after',
                                                         'at_or_after',
                                                         'before',
                                                         'at_or_before',
                                                         'between'],
                                                'description': "수업의 시작 시각 기준: exact=정각 시작(==), before='이전' 미만(<), "
                                                               "at_or_before='까지' 이하(<=), after='초과' 기준 시각 제외(>), "
                                                               "at_or_after='부터' 이상(>=), between='A부터 B까지' 시작 시각 "
                                                               '구간 양끝 포함입니다. 예: 3시 이전은 15:00 제외, 3시부터는 15:00 포함. '
                                                               '생략하면 exact입니다.'}},
                 'required': [],
                 'additionalProperties': False}},
 {'type': 'function',
  'name': 'get_my_memberships',
  'description': '로그인한 회원 본인의 보유 이용권, 상품 유형, 유효기간, 남은 횟수를 조회합니다. 단일 및 콤보 이용권을 모두 반환하며 콤보 상품은 '
                 "remaining_sessions_by_type에서 '7:1 레슨'(그룹), '2:1 듀엣 레슨'(듀엣), '1:1 개인 레슨'(개인)의 남은 횟수를 각각 확인합니다. "
                 '상품 구매 가능 목록은 get_membership_info, 실제 빈자리는 get_available_classes, 이미 예약한 날짜는 get_my_bookings로 '
                 '확인합니다. 로그인 회원은 내부에서 결정하며 user_id 인자는 받지 않습니다.',
  'parameters': {'type': 'object', 'properties': {}, 'required': [], 'additionalProperties': False}},
 {'type': 'function',
  'name': 'get_my_bookings',
  'description': '로그인 회원 본인의 실제 수업 예약 날짜와 강사, 레슨 유형, 예약 상태를 조회합니다. 그룹=7:1 레슨, 듀엣=2:1 듀엣 레슨, 개인=1:1 개인 레슨으로 해석하며 '
                 '콤보는 예약 수업 유형이 아닙니다. CONFIRMED=확정, WAITLISTED=대기, CANCELLED=취소, COMPLETED=완료를 구분합니다. 본인 예약 내역과 '
                 '같은 날짜 중복 예약 여부를 확인할 때 사용하고, 전체 수업 빈자리는 get_available_classes로 조회합니다. 예약 생성·취소 기능은 없으며 user_id '
                 '인자는 받지 않습니다.',
  'parameters': {'type': 'object',
                 'properties': {'status': {'type': 'string',
                                           'enum': ['CONFIRMED', 'WAITLISTED', 'CANCELLED', 'COMPLETED'],
                                           'description': '예약 상태. CONFIRMED=예약 확정, WAITLISTED=예약 대기, CANCELLED=예약 '
                                                          '취소, COMPLETED=수업 완료. 생략하면 전체 상태 조회.'},
                                'week_status': {'type': 'string',
                                                'enum': ['ACTIVE', 'INACTIVE'],
                                                'description': 'ACTIVE=현재 관리 중인 주차, INACTIVE=지난 주차. 생략하면 전체 주차 '
                                                               '조회.'},
                                'day_of_week': {'type': 'string',
                                                'enum': ['월요일', '화요일', '수요일', '목요일', '금요일'],
                                                'description': "예약 내역의 단일 요일 필터입니다. '월·수'는 각각 호출합니다. 생략하면 전체 "
                                                               '요일입니다.'}},
                 'required': [],
                 'additionalProperties': False}},
 {'type': 'function',
  'name': 'search_policy_info',
  'description': "센터의 예약 오픈, 취소, 대기, 출석, 환불 및 운영시간에 관한 정책과 약관을 조회합니다. '대기 예약 가능해?', '취소하면 횟수 돌려받아?'처럼 규정을 묻는 질문에 "
                 '사용합니다. 실제 빈자리나 회원 본인의 예약 내역을 조회하는 함수는 아닙니다.',
  'parameters': {'type': 'object',
                 'properties': {'keyword': {'type': 'string',
                                            'description': "정책 검색어(예: '예약', '취소', '대기', '출석', '환불', '운영')입니다. '대기 "
                                                           "예약'이면 '대기'처럼 핵심어를 전달합니다. 생략하면 모든 정책을 조회합니다."}},
                 'required': [],
                 'additionalProperties': False}}]

print(json.dumps(tools, ensure_ascii=False, indent=2))

[
  {
    "type": "function",
    "name": "get_membership_info",
    "description": "센터에 등록된 이용권 상품의 가격, 전체 이용 횟수, 유효기간, 구성 정보를 조회합니다. 전체 이용 횟수(total_sessions)는 sessions로 검색하며 SINGLE과 COMBO 모두 적용됩니다. SINGLE 상품은 lesson_type과 sessions로 검색할 수 있습니다. COMBO 상품은 product_type='COMBO'로 검색하고 반환된 components에서 구성별 레슨 유형과 횟수를 확인합니다. 콤보의 모든 구성과 횟수가 명시되어 정확한 전체 횟수를 알 수 있다면 합계를 sessions로 전달합니다. 일부 구성만 알려졌거나 특정 구성이 포함된 모든 상품을 찾을 때는 sessions를 생략합니다. 본인의 보유 이용권과 잔여 횟수는 get_my_memberships로 확인합니다. 조회 결과의 status를 확인하며, 반환된 모든 상품이 판매 중이라고 가정하지 않습니다.",
    "parameters": {
      "type": "object",
      "properties": {
        "sessions": {
          "type": "integer",
          "description": "상품의 전체 이용 횟수(total_sessions)에 정확히 일치하는 정수입니다. SINGLE과 COMBO 상품 모두에 적용됩니다. '6회권 있어?'는 sessions=6으로 검색합니다. '총 60회 콤보권 있어?'는 sessions=60, product_type='COMBO'로 검색합니다. 콤보 상품에서 전체 구성을 모두 알 수 있고 그 구성과 정확히 일치하는 상품을 찾는다면, 각 구성의 sessions를 합산하여 전달합니다. 예: '개인 6회 + 그룹 32회 콤보권'은 6+32=38이므로 sessions=38, product_type='COMBO'입니다. 예: '개인

In [39]:
tools_return_format = [
    # 2-1. 이용권 상품 정보 조회
    {
        "function_name": "get_membership_info",
        "result_columns_format": {
            "success": "boolean",
            "membership_count": "integer",
            "membership_info": "list(dict[membership_id: string, membership_name: string, product_type: string, lesson_type: string, duration_months: integer, total_sessions: integer, sessions_per_month: integer or null, capacity: integer or null, price: integer, price_per_session: integer or null, components: list(dict[lesson_type: string, sessions: integer]), status: string, original_price: integer (optional), bundle_discount: integer (optional)])",
            "message": "string (조회 결과가 없을 때)"
        }
    },

    # 2-2. 수업 시간표 조회
    {
        "function_name": "get_class_schedule",
        "result_columns_format": {
            "success": "boolean",
            "class_count": "integer",
            "class_schedule": "list(dict[class_id: string, day_of_week: string, start_time: string(HH:MM), end_time: string(HH:MM), lesson_type: string, instructor_id: string, instructor_name: string, capacity: integer, status: string])",
            "message": "string (조회 오류 또는 결과가 없을 때)"
        }
    },

    # 2-3. 예약 가능한 수업 조회
    {
        "function_name": "get_available_classes",
        "result_columns_format": {
            "success": "boolean",
            "week_start_date": "string(%Y-%m-%d)",
            "available_class_count": "integer",
            "available_classes": "list(dict[class_id: string, class_date: string(%Y-%m-%d), day_of_week: string, start_time: string(HH:MM), end_time: string(HH:MM), lesson_type: string, instructor_name: string, capacity: integer, booked_count: integer, remaining_seats: integer])",
            "message": "string"
        }
    },

    # 2-4. 내 보유 이용권 조회
    {
        "function_name": "get_my_memberships",
        "result_columns_format": {
            "success": "boolean",
            "user_id": "string",
            "user_name": "string",
            "reference_date": "string(%Y-%m-%d)",
            "membership_count": "integer",
            "user_memberships": "list(dict[user_membership_id: string, membership_id: string, membership_name: string, product_type: string, lesson_type: string, components: list(dict[lesson_type: string, sessions: integer]), total_sessions: integer, start_date: string(%Y-%m-%d), expires_at: string(%Y-%m-%d), remaining_sessions: integer, remaining_sessions_by_type: dict(string: integer), status: string, is_usable: boolean])",
            "message": "string"
        }
    },

    # 2-5. 내 예약 내역 조회
    {
        "function_name": "get_my_bookings",
        "result_columns_format": {
            "success": "boolean",
            "user_id": "string",
            "user_name": "string",
            "booking_count": "integer",
            "bookings": "list(dict[booking_id: string, class_id: string, class_date: string(%Y-%m-%d), day_of_week: string, start_time: string(HH:MM), end_time: string(HH:MM), lesson_type: string, instructor_name: string, booked_at: string(%Y-%m-%d %H:%M), status: string, week_status: string, waitlist_position: integer or null, cancelled_at: string(%Y-%m-%d %H:%M) or null, user_membership_id: string or null, credit_deducted: boolean, credit_restored: boolean])",
            "message": "string"
        }
    },

    # 2-6. 센터 정책 및 약관 조회
    {
        "function_name": "search_policy_info",
        "result_columns_format": {
            "success": "boolean",
            "policy_count": "integer",
            "policies": "list(dict[policy_id: string, keyword: string, content: string, effective_date: string(%Y-%m-%d)])",
            "message": "string (조회 결과가 없을 때)"
        }
    }
]

# 반환 포맷 확인
print(json.dumps(tools_return_format, ensure_ascii=False, indent=2))

[
  {
    "function_name": "get_membership_info",
    "result_columns_format": {
      "success": "boolean",
      "membership_count": "integer",
      "membership_info": "list(dict[membership_id: string, membership_name: string, product_type: string, lesson_type: string, duration_months: integer, total_sessions: integer, sessions_per_month: integer or null, capacity: integer or null, price: integer, price_per_session: integer or null, components: list(dict[lesson_type: string, sessions: integer]), status: string, original_price: integer (optional), bundle_discount: integer (optional)])",
      "message": "string (조회 결과가 없을 때)"
    }
  },
  {
    "function_name": "get_class_schedule",
    "result_columns_format": {
      "success": "boolean",
      "class_count": "integer",
      "class_schedule": "list(dict[class_id: string, day_of_week: string, start_time: string(HH:MM), end_time: string(HH:MM), lesson_type: string, instructor_id: string, instructor_name: string, capacity: integer, s

## **5. Scenarios**

In [40]:
# 1. 현재 6개 함수로 처리 가능한 시나리오
question_topics = [

    # 1. 이용권 상품 정보 조회
    "센터에서 판매하는 전체 이용권 상품 조회",
    "특정 횟수의 이용권 상품 조회",
    "단일(SINGLE) 이용권 종류 및 가격 조회",
    "레슨 유형별 이용권 상품 검색",
    "이용권 유효기간 및 전체 이용 횟수 확인",
    "서로 다른 이용권의 가격과 구성 비교",
    "콤보(COMBO) 이용권 전체 상품 조회",
    "콤보 이용권의 레슨별 구성과 횟수 조회",
    "특정 레슨 구성과 횟수가 정확히 일치하는 콤보 상품 검색",
    "특정 레슨 횟수가 포함된 단일 및 콤보 상품 검색",

    # 2. 정규 수업 시간표 조회
    "전체 정규 수업 시간표 조회",
    "특정 요일의 수업 시간표 조회",
    "특정 강사의 정규 수업 시간표 조회",
    "강사와 요일 조건을 결합한 시간표 조회",
    "레슨 유형별 정규 수업 시간표 조회",
    "특정 시작 시각의 수업 조회",
    "특정 시각 이전 또는 이후의 수업 조회",
    "특정 시간 범위 내 수업 조회",
    "월요일과 수요일 등 여러 요일의 시간표 조회",

    # 3. 예약 가능한 수업 및 잔여석 조회
    "예약 오픈된 주차의 예약 가능 수업 조회",
    "특정 요일의 잔여석 있는 수업 조회",
    "특정 강사의 예약 가능한 수업 조회",
    "레슨 유형별 예약 가능한 수업 조회",
    "특정 시간대의 예약 가능한 수업 조회",
    "수업 날짜별 예약 인원 및 잔여석 조회",
    "월요일과 수요일의 예약 가능 수업 비교",

    # 4. 로그인 회원의 보유 이용권 조회
    "내가 보유한 전체 이용권 조회",
    "내 이용권의 남은 사용 횟수 조회",
    "내 이용권의 시작일과 만료일 조회",
    "내 콤보 이용권의 레슨 유형별 잔여 횟수 조회",
    "내 이용권의 사용 가능 상태 확인",

    # 5. 로그인 회원의 예약 내역 조회
    "내 전체 수업 예약 내역 조회",
    "내 확정 예약 내역 조회",
    "내 예약 대기 상태 및 대기 순번 조회",
    "내 취소된 예약 내역 조회",
    "내 완료된 예약 내역 조회",
    "내 특정 요일의 예약 내역 조회",
    "내 현재 관리 중인 주차의 예약 내역 조회",
    "특정 날짜에 이미 예약한 수업이 있는지 확인",

    # 6. 센터 운영 정책 및 약관 조회
    "수업 예약 오픈 일정과 규정 조회",
    "수업 취소 마감 시간 및 규정 조회",
    "수업 취소 시 이용권 횟수 복구 규정 조회",
    "예약 대기 신청 관련 규정 조회",
    "결석 및 출석 관련 규정 조회",
    "이용권 환불 규정 조회",
    "센터 운영시간 및 기타 약관 조회",

    # 7. 복합 Function Calling
    "정규 수업 시간표와 실제 예약 가능 수업 함께 조회",
    "강사별 시간표와 잔여석 함께 조회",
    "여러 요일의 수업 시간표와 잔여석 비교",
    "내 이용권으로 이용할 수 있는 레슨 유형 확인 후 수업 조회",
    "내 이용권과 수업 잔여석을 종합한 예약 가능 여부 확인",
    "기존 예약과 이용권 및 잔여석을 종합한 조건부 예약 가능 여부 확인",
    "내 예약 내역과 취소 규정을 함께 조회",
    "내 이용권 만료일과 이용 가능한 수업 날짜 비교"
]

In [41]:
# 2. 현재 함수로 처리 불가능한 시나리오
unsupported_scenarios = [

    # 1. 예약 생성 및 변경
    "그룹 수업 예약을 실제로 생성해 달라는 요청",
    "개인 레슨 예약을 실제로 생성해 달라는 요청",
    "기존 수업 예약을 취소해 달라는 요청",
    "예약한 수업의 날짜나 시간을 변경해 달라는 요청",
    "예약한 수업의 담당 강사를 변경해 달라는 요청",
    "만석 수업에 대기 신청을 등록해 달라는 요청",
    "예약 대기를 취소해 달라는 요청",

    # 2. 이용권 구매 및 변경
    "새 이용권을 실제로 구매하거나 결제해 달라는 요청",
    "이용권을 다른 상품으로 변경해 달라는 요청",
    "이용권 유효기간을 연장해 달라는 요청",
    "이용권 잔여 횟수를 직접 수정해 달라는 요청",
    "이용권을 다른 회원에게 양도해 달라는 요청",
    "이용권 환불을 실제로 신청하거나 처리해 달라는 요청",
    "환불 금액을 실제로 지급해 달라는 요청",

    # 3. 출석 및 이용 통계
    "수업 출석을 직접 등록해 달라는 요청",
    "출석 상태를 결석이나 지각으로 변경해 달라는 요청",
    "실제 출석 기록을 기준으로 출석률을 계산해 달라는 요청",
    "월별 실제 출석 횟수와 결석 횟수를 통계로 조회하는 요청",
    "누적 방문 횟수 통계를 조회하는 요청",

    # 4. 회원 정보 관리(보안상의 관리)
    "다른 회원의 이용권이나 예약 내역 조회",
    "회원 이름이나 전화번호 변경",
    "비밀번호 변경 및 재설정",
    "회원 탈퇴 요청",
    "다른 회원의 계정으로 예약을 생성하는 요청",

    # 5. 현재 조회 함수의 범위 밖
    "현재 예약 오픈 주차 외의 미래 수업 잔여석 확정 조회",
    "만석인 모든 수업의 예약 인원과 대기 현황 조회",
    "전체 회원의 대기 명단과 대기 순서 조회",
    "강사의 비공개 일정이나 휴무 사유 조회",
    "센터의 실제 실시간 결제 내역 조회",
    "알림톡이나 SMS 발송",
    "예약 알림 및 대기 순번 변경 알림 등록",
    "상담 요청을 실제로 담당 직원에게 전송",
    "영수증이나 결제 증빙 서류 발급",

    # 6. 센터와 무관한 외부 요청
    "실시간 날씨 조회",
    "주식 가격 및 환율 조회",
    "음식 주문 및 배달 요청",
    "외부 병원이나 다른 센터의 예약 관리"
]

In [42]:
# 일상 데이터 시나리오 가져오기
question_df = pd.read_csv('https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv')
question_df = question_df.drop_duplicates(subset='Q')
chit_chat_question = question_df['Q'].to_list()
questions = ['"' + question +'"과 같은 사용자의 단순한 잡담' for question in chit_chat_question]
questions[:5]

['"12시 땡!"과 같은 사용자의 단순한 잡담',
 '"1지망 학교 떨어졌어"과 같은 사용자의 단순한 잡담',
 '"3박4일 놀러가고 싶다"과 같은 사용자의 단순한 잡담',
 '"3박4일 정도 놀러가고 싶다"과 같은 사용자의 단순한 잡담',
 '"PPL 심하네"과 같은 사용자의 단순한 잡담']

## **6. Data Generation — 2단계 연속 생성 (전체 5~6턴 유연 검증)**

In [47]:
# 원본 함수·DB·Tool Schema는 건드리지 않음. 생성용 프롬프트와 6턴 행동 계획만 V2로 개선
scenario_types = {
    "TOOL_CALL": "조회가 가능하면 필요한 tool을 즉시 호출 (선택 인자가 비어도 전체 조회 가능)",
    "CLARIFY": "필수 정보가 Tool/문맥으로도 확보되지 않거나 조건이 충돌할 때만 한 가지를 되물음",
    "FOLLOW_UP": "앞선 tool 결과에 있는 정보는 재호출 없이 답변",
    "DIRECT_ANSWER": "인사·잡담·일반 설명이나 이미 제공된 정보만으로 답할 수 있으면 바로 답변",
    "UNSUPPORTED": "예약 생성·취소·결제 같은 미지원 작업은 했다고 거짓 주장하지 않음",
    "MULTI_CALL": "서로 다른 조건/함수를 함께 확인할 필요가 있으면 여러 tool_call 수행"
}

# 실제 챗봇과 동일하게 적용하는 행동 경계 원칙 (의도 분류 모델을 별도로 만들지 않음)
DECISION_RULES = """[행동 판단 순서]
A. 요청이 단순 인사·일반 설명이거나 이미 대화에 제공된 정보로 답할 수 있으면 DIRECT_ANSWER/FOLLOW_UP.
B. 최신 센터/회원 DB 값이 필요하면 Tool을 사용. 누락된 값이 선택 파라미터라면 되묻지 말고 허용된 범위에서 전체 조회.
C. 정보가 부족해도 현재 Tool로 직접 알아낼 수 있으면 Tool을 사용한다. 회원에게 DB에 있는 잔여 횟수나 시간표를 거꾸로 묻지 않는다.
D. CLARIFY는 사용자의 의도가 여러 개로 해석되어 잘못된 대상을 고를 위험이 있고, 대화 문맥·기본 조회·Tool로도 해소할 수 없을 때만 선택.
   - 필요한 구체 정보 한 가지만 질문하고 같은 질문을 반복하지 않는다.
   - 사용자가 답해 모호함이 풀리면 즉시 필요한 Tool 호출/직접 답변으로 이어간다.
E. 조건이 여러 개지만 서로 양립 가능하면 MULTI_CALL. 복수 요일 요청을 하나로 임의 축소하거나 CLARIFY 하지 않는다.
F. 진짜 미지원 기능만 기존 UNSUPPORTED 행동으로 안내. 미지원 비율을 추가하지 않는다.
G. FUNCTION_CALL 전 안내 멘트 없이 바로 호출하고, 이후 실제 Tool 결과를 확인한 다음 자연어 답변을 생성한다.

[비슷해 보이지만 정답 행동이 다른 대비 예시 — 형식만 참조하고 문구 반복 금지]
- '수업 시간표 알려줘' → TOOL_CALL: get_class_schedule({}) (요일/강사 생략은 괜찮음).
- '월요일 그룹 수업 시간표 알려줘' → TOOL_CALL: 지정 조건으로 조회.
- '월요일 빈자리 남았어?' → TOOL_CALL: get_available_classes (정규 시간표와 다른 도구).
- '내 이용권 얼마나 남았어?' → TOOL_CALL: get_my_memberships({}) (잔여 횟수를 사용자에게 묻지 않음).
- '7:1 듀엣 수업 알려줘' → CLARIFY: 그룹 7:1과 듀엣 2:1 중 어느 쪽인지 한 가지만 질문.
- '그 수업 보여줘' + 이전 대화에서 수업 하나만 지칭됨 → 필요한 새 조회만 실행하거나 이전 결과로 답변.
- '그 수업 보여줘' + 이전 대화에 서로 다른 후보 두 개이고 무엇을 가리키는지 불명확 → CLARIFY.
- '월요일이랑 수요일 수업 알려줘' → MULTI_CALL: 요일별 조회; 선택을 강요하지 않는다.
- '아까 알려준 수업 종료 시각 다시 알려줘' + 이전 결과에 종료 시간이 있음 → FOLLOW_UP: 새 호출 없이 답변.
- '그룹 레슨과 듀엣의 일반적 차이가 뭐야?' → DIRECT_ANSWER: 일반적 개념 설명 가능. 실제 가격/시간표를 단정하지 않는다.
- '내 이용권으로 가능한 수업 알려줘' → 필요한 경우 get_my_memberships + get_available_classes 등 MULTI_CALL.
"""

# 고객 발화는 5~6개. 마지막 턴은 문의 종료로 고정하되 이전 턴에서 행동 다양성 확보
# CLARIFY 다음 고객 발화가 모호함을 해소하면 즉시 TOOL_CALL
ACTION_PLANS = [
    ["TOOL_CALL", "FOLLOW_UP", "CLARIFY", "TOOL_CALL", "UNSUPPORTED", "MULTI_CALL"],
    ["DIRECT_ANSWER", "TOOL_CALL", "FOLLOW_UP", "CLARIFY", "TOOL_CALL", "UNSUPPORTED"],
    ["TOOL_CALL", "CLARIFY", "TOOL_CALL", "MULTI_CALL", "FOLLOW_UP", "DIRECT_ANSWER"],
    ["TOOL_CALL", "FOLLOW_UP", "MULTI_CALL", "UNSUPPORTED", "DIRECT_ANSWER", "TOOL_CALL"],
    ["CLARIFY", "TOOL_CALL", "FOLLOW_UP", "TOOL_CALL", "DIRECT_ANSWER", "UNSUPPORTED"],
    ["TOOL_CALL", "MULTI_CALL", "FOLLOW_UP", "CLARIFY", "TOOL_CALL", "UNSUPPORTED"],
    ["DIRECT_ANSWER", "TOOL_CALL", "MULTI_CALL", "FOLLOW_UP", "UNSUPPORTED", "DIRECT_ANSWER"],
    ["TOOL_CALL", "FOLLOW_UP", "DIRECT_ANSWER", "MULTI_CALL", "CLARIFY", "TOOL_CALL"],
]
TOPIC_GROUPS = {
    "이용권 상품": question_topics[:10],
    "정규 시간표": question_topics[10:19],
    "실제 빈자리": question_topics[19:26],
    "내 보유 이용권": question_topics[26:31],
    "내 예약 내역": question_topics[31:39],
    "운영 정책": question_topics[39:46],
    "복합 조회": question_topics[46:]
}
SPEECH_STYLES = [
    "간단한 구어체와 존댓말", "친근한 반말·존댓말 혼용", "오타가 조금 있는 짧은 질문",
    "구체적 요일·시간 조건", "일부 조건을 나중에 수정", "문맥상 지시어와 후속 질문",
    "두 조건을 비교", "예약 가능과 정규 시간표를 명확히 구별",
    "기존 조회 결과를 다시 질문", "계좌·개인정보 등 미지원 요청 거절",
]

def make_generation_plan(index):
    categories = list(TOPIC_GROUPS)
    main = categories[index % len(categories)]
    other = categories[(index // len(categories) + 2) % len(categories)]
    turn_count = 5 if index % 2 else 6
    # 기존 6가지 행동 유형은 앞선 턴에서 순환 학습하고, 마지막 턴은 종료 대화로 배치
    action_prefix = ACTION_PLANS[index % len(ACTION_PLANS)][:turn_count - 1]
    if action_prefix[-1] == "CLARIFY":
        # 사용자가 해소하지 않은 CLARIFY로 상담이 끝나지 않도록 마지막 조회/답변으로 대체
        action_prefix[-1] = "TOOL_CALL"
    return {
        "turn_count": turn_count,
        "actions": action_prefix + ["CLOSE"],
        "topics": [random.choice(TOPIC_GROUPS[main]), random.choice(TOPIC_GROUPS[other])],
        "style": SPEECH_STYLES[index % len(SPEECH_STYLES)],
        "clarify_case": ["모순된 레슨 유형(7:1 듀엣)", "참조 대상이 없는 '그 수업'",
                         "앞선 맥락에 후보가 둘인 '그 둘 중 하나'", "맥락에 후보가 여럿인 강사/수업"][(index // 4) % 4],
    }

def make_function_calling_data(tools, cids, tools_return_format, question_topics,
                               unsupported_scenarios, plan=None):
    # 무한정 API 응답을 기다리지 않고, SDK 내부 자동 재시도를 제한
    client = openai.OpenAI(timeout=120.0, max_retries=1)
    cid = random.choice(cids)
    plan = plan or make_generation_plan(random.randrange(10000))
    unsupported_example = random.choice(unsupported_scenarios)
    prompt = f"""당신은 글로리 필라테스 상담용 Qwen Function Calling SFT 대화를 생성합니다.
회원 ID: {cid}, 대화 날짜: {CHAT_DATE}
Tool 명세(JSON): {json.dumps(tools, ensure_ascii=False)}
함수 반환 스키마 참고: {json.dumps(tools_return_format, ensure_ascii=False)}
조회 주제(표현·요청은 자유롭게 변형): {plan['topics']}
{plan['turn_count']}턴별 목표 행동(마지막 CLOSE 포함): {plan['actions']}
말투 다양성: {plan['style']}
모호한 질문 소재(필요 시): {plan['clarify_case']}
처리 불가능한 요청 소재(필요 시): {unsupported_example}

{DECISION_RULES}
주의: 행동 계획을 만족하도록 사용자의 질문 자체도 알맞게 구성한다. TOOL_CALL 턴을 만들면서 모호한 사용자 질문을 억지로 붙이거나, CLARIFY 턴에 전체 조회가 가능한 질문을 넣지 않는다. UNSUPPORTED 행동은 앞선 턴 중 지정된 경우에만 사용한다. 마지막 CLOSE는 추가 문의가 없다고 밝히는 종료 턴이며 Tool 호출 없이 마무리한다.

[행동 결정 원칙 — 절대 지킬 것]
1. 고객 발화는 지정된 5개 또는 6개, 각각에 자연어 최종 (AI 상담사) 응답 1개. function_call/function_response는 고객 턴으로 계산하지 않는다. 일상 표현은 반복하지 말고, 실제 필라테스 상담답게 자연스럽게 변형.
2. TOOL_CALL: 사용자 요청을 현재 Tool로 수행할 수 있다면 고객 바로 다음에 (function_call) -> (function_response) -> (AI 상담사). 호출 전 '확인해 드릴게요', '조회하겠습니다' 같은 안내 assistant 발화 절대 금지.
3. 선택 파라미터가 빠져도 전체/기본 조회가 되는 함수는 바로 실행. 예를 들어 '수업 시간표 알려줘'는 get_class_schedule({{}}), '빈자리 알려줘'는 get_available_classes({{}}), '내 이용권 남은 횟수'는 get_my_memberships({{}}). 불필요하게 강사·요일·레슨 종류를 물어보지 말기.
4. CLARIFY: 문맥에 여러 후보가 있어 대상을 특정할 수 없거나 조건이 논리적으로 충돌할 때만 자연어로 **딱 한 가지** 추가 질문. 이때 함수 호출 없음. 다음 고객 답으로 해소되면 즉시 함수 호출. 단, '월/수 요일 모두'처럼 복수 요청은 모호함이 아니므로 두 개 호출.
5. FOLLOW_UP: 이전 조회 결과에 있는 사실 또는 이전 사용자 발화만으로 답할 수 있으면 함수 없이 자연어로 답변. 이전 결과에 없는 최신 값이 필요하면 조회한다.
6. DIRECT_ANSWER: 인사/감사/일반적 레슨 비교·개념 설명/이미 충분히 제공된 정보는 자연어로 답한다. 실제 가격·잔여석·개인 예약 현황은 임의로 지어내지 않는다.
7. UNSUPPORTED: 현재 6개 Tool에 없는 예약 생성/취소/변경/결제/환불 실행, 타인의 정보 접근 등을 절대 실행했다고 말하지 말 것. 필요하면 가능 범위를 간결히 안내.
8. MULTI_CALL: 두 개 이상 실제 함수 호출이 필요한 요청. 한 (function_call) 리스트 안에 모든 호출을 넣고, 같은 순서의 (function_response) 리스트를 제공. 없는 함수는 생성하지 않기.
9. function_call name 및 arguments는 실제 Tool Schema와 타입, enum을 정확히 준수. 필요 없는 인자는 넣지 마라. 정규 시간표와 실제 예약 가능 수업을 구별하라.
10. function_response는 **함수마다 서로 다른 tools_return_format의 result_columns_format**을 정확히 따라 JSON으로 작성한다. success, 개수, 리스트 등 해당 결과에 필요한 필드 및 리스트 내부 객체 필드를 생략하지 않는다. message처럼 결과 없음/오류에서만 사용하는 조건부 필드는 해당 상황일 때 넣는다. 결과 리스트(class_schedule, membership_info, bookings 등)의 레코드 수를 일괄적으로 3개 이하로 제한하지 않는다. 각 함수가 반환하는 필드와 리스트 내부 필드를 완전한 JSON으로 작성하고, 건수 필드는 실제 제시한 리스트의 길이와 일치시킨다. MULTI_CALL이면 호출마다 정확히 하나의 response 객체를 같은 순서로 제공한다. 이후 별도 파이썬 단계에서 실제 DB로 검증/교체하고 답변도 검증/재생성한다.
11. 첫 고객 발화 전 [고객 ID] / [대화날짜]를 제공. 출력은 대화 원문만, 코드블록/해설/행동 라벨(TOOL_CALL 등)은 출력하지 말 것.
12. (function_call)과 (function_response)는 해당 턴에만 배치, 연속된 (AI 상담사) 발화 금지. 함수 호출이 없는 턴은 (고객) → (AI 상담사). 함수가 단 1개여도 (function_call)과 (function_response)는 반드시 JSON 리스트 형식인 [{...}]로 출력한다.
13. CLARIFY 바로 다음 목표가 TOOL_CALL이면 다음 고객 발화는 필요한 정보를 답변하는 내용이어야 한다.
14. 마지막 행동 CLOSE: 마지막 (고객) 발화에서는 "더 궁금한 건 없어요. 감사합니다!"처럼 추가 질문이 없음을 자연스럽게 알린다. 마지막 (AI 상담사)는 문의가 해결된 것을 확인하고 "더 궁금하신 사항이 없으시면 상담을 종료하겠습니다. 감사합니다!!"처럼 종료 인사로 마무리한다. CLOSE에는 function_call/function_response를 생성하지 않는다. 고객 마지막 질문이 아직 해결되지 않았는데 상담을 임의 종료하지 않는다.
15. 마지막 (AI 상담사) 답변의 맨 끝은 반드시 "감사합니다!!"로 끝낸다. 다른 일반 응답에서는 상담 종료 인사를 반복하지 않는다.

[상세 출력 예시]
아래 4가지 예시는 각각 고객 발화 6개이며 함수별 자세한 function_response JSON을 보여주는 참고 사례이다. 실제 데이터는 지정된 5~6턴으로 생성한다.
예시 4처럼 조건이 명백히 충돌하거나 문맥에 후보가 여럿이라 지시 대상을 확정할 수 없을 때만 CLARIFY를 한다.
기존 예시에서 전체 조회가 가능한데 요일, 강사 등을 억지로 질문하던 방식은 따르지 않는다.
예시는 형식 참고용이고 실제 생성 시에는 반드시 위의 여섯 행동 계획 순서를 우선하며, 질문과 엔터티는 새로 작성한다.
"""

    prompt += r"""**예시 1 | 이용권 조회 + 월·수 2건 조회 + FOLLOW_UP (6턴)**
[고객 ID] U001
[대화날짜] 2026-11-02
(고객) 안녕하세요.
(AI 상담사) 안녕하세요! 글로리 필라테스 AI 상담사입니다. 무엇을 도와드릴까요?
(고객) 지금 그룹 이용권 몇 회 남았어요?
(function_call) [{"name": "get_my_memberships", "arguments": {}}]
(function_response) [{"success": true, "user_id": "U001", "user_name": "장현우", "reference_date": "2026-11-02", "membership_count": 2, "user_memberships": [{"user_membership_id": "M001", "membership_id": "P003", "membership_name": "7:1 그룹 96회권", "product_type": "SINGLE", "lesson_type": "7:1 레슨", "components": [{"lesson_type": "7:1 레슨", "sessions": 96}], "total_sessions": 96, "start_date": "2026-09-01", "expires_at": "2027-08-31", "remaining_sessions": 93, "remaining_sessions_by_type": {"7:1 레슨": 93}, "status": "ACTIVE", "is_usable": true}, {"user_membership_id": "M002", "membership_id": "P004", "membership_name": "2:1 듀엣 12회권", "product_type": "SINGLE", "lesson_type": "2:1 듀엣 레슨", "components": [{"lesson_type": "2:1 듀엣 레슨", "sessions": 12}], "total_sessions": 12, "start_date": "2026-09-01", "expires_at": "2026-10-31", "remaining_sessions": 12, "remaining_sessions_by_type": {"2:1 듀엣 레슨": 12}, "status": "ACTIVE", "is_usable": false}], "message": "내 이용권 조회 완료"}]
(AI 상담사) 보유 이용권은 7:1 그룹 96회권 잔여 93회(만료 2027-08-31, 사용 가능), 2:1 듀엣 12회권 잔여 12회(만료 2026-10-31, 사용 불가)입니다.
(고객) 그렇군요. 그럼 월요일이랑 수요일 7:1 시간표를 모두 보여주세요. 한 번에 정리해 주세요.
(function_call) [{"name": "get_class_schedule", "arguments": {"day_of_week": "월요일", "lesson_type": "7:1 레슨"}}, {"name": "get_class_schedule", "arguments": {"day_of_week": "수요일", "lesson_type": "7:1 레슨"}}]
(function_response) [{"success": true, "class_count": 2, "class_schedule": [{"class_id": "CL043", "day_of_week": "월요일", "start_time": "08:00", "end_time": "08:50", "lesson_type": "7:1 레슨", "instructor_id": "I001", "instructor_name": "서다은", "capacity": 7, "status": "ACTIVE"}, {"class_id": "CL072", "day_of_week": "월요일", "start_time": "18:00", "end_time": "18:50", "lesson_type": "7:1 레슨", "instructor_id": "I001", "instructor_name": "서다은", "capacity": 7, "status": "ACTIVE"}]}, {"success": true, "class_count": 1, "class_schedule": [{"class_id": "CL078", "day_of_week": "수요일", "start_time": "20:00", "end_time": "20:50", "lesson_type": "7:1 레슨", "instructor_id": "I001", "instructor_name": "서다은", "capacity": 7, "status": "ACTIVE"}]}]
(AI 상담사) 정규 시간표: 월요일 08:00~08:50 서다은 강사님 7:1 레슨(CL043), 월요일 18:00~18:50 서다은 강사님 7:1 레슨(CL072). 정규 시간표: 수요일 20:00~20:50 서다은 강사님 7:1 레슨(CL078). 정규 시간표는 예약 가능한 잔여석을 의미하지 않습니다.
(고객) 알겠어요. 아까 그룹 이용권은 그대로 이용할 수 있는 거죠?
(AI 상담사) 네, 앞서 확인한 7:1 그룹 이용권은 2027년 8월 31일까지 유효하며 93회가 남아 있습니다.
(고객) 만료일은 언제였나요?
(AI 상담사) 2027년 8월 31일입니다.
(고객) 감사합니다. 이제 궁금한 건 다 해결됐어요.
(AI 상담사) 추가로 궁금하신 사항이 없으시면 상담을 종료하겠습니다. 감사합니다!!

**예시 2 | 순차 조회 + 서로 다른 2개 함수 조회 + FOLLOW_UP (6턴)**
[고객 ID] U001
[대화날짜] 2026-11-02
(고객) 안녕하세요.
(AI 상담사) 안녕하세요! 글로리 필라테스 AI 상담사입니다. 무엇을 도와드릴까요?
(고객) 제 그룹 이용권으로 월요일 저녁 6시 이후에 예약할 수 있는 수업이 있나요?
(function_call) [{"name": "get_my_memberships", "arguments": {}}, {"name": "get_available_classes", "arguments": {"day_of_week": "월요일", "lesson_type": "7:1 레슨", "start_time": "18:00", "time_filter": "at_or_after"}}]
(function_response) [{"success": true, "user_id": "U001", "user_name": "장현우", "reference_date": "2026-11-02", "membership_count": 2, "user_memberships": [{"user_membership_id": "M001", "membership_id": "P003", "membership_name": "7:1 그룹 96회권", "product_type": "SINGLE", "lesson_type": "7:1 레슨", "components": [{"lesson_type": "7:1 레슨", "sessions": 96}], "total_sessions": 96, "start_date": "2026-09-01", "expires_at": "2027-08-31", "remaining_sessions": 93, "remaining_sessions_by_type": {"7:1 레슨": 93}, "status": "ACTIVE", "is_usable": true}, {"user_membership_id": "M002", "membership_id": "P004", "membership_name": "2:1 듀엣 12회권", "product_type": "SINGLE", "lesson_type": "2:1 듀엣 레슨", "components": [{"lesson_type": "2:1 듀엣 레슨", "sessions": 12}], "total_sessions": 12, "start_date": "2026-09-01", "expires_at": "2026-10-31", "remaining_sessions": 12, "remaining_sessions_by_type": {"2:1 듀엣 레슨": 12}, "status": "ACTIVE", "is_usable": false}], "message": "내 이용권 조회 완료"}, {"success": true, "week_start_date": "2026-11-02", "available_class_count": 5, "available_classes": [{"class_id": "CL072", "class_date": "2026-11-02", "day_of_week": "월요일", "start_time": "18:00", "end_time": "18:50", "lesson_type": "7:1 레슨", "instructor_name": "서다은", "capacity": 7, "booked_count": 0, "remaining_seats": 7}, {"class_id": "CL017", "class_date": "2026-11-02", "day_of_week": "월요일", "start_time": "20:00", "end_time": "20:50", "lesson_type": "7:1 레슨", "instructor_name": "배수현", "capacity": 7, "booked_count": 2, "remaining_seats": 5}, {"class_id": "CL032", "class_date": "2026-11-02", "day_of_week": "월요일", "start_time": "20:00", "end_time": "20:50", "lesson_type": "7:1 레슨", "instructor_name": "임채아", "capacity": 7, "booked_count": 1, "remaining_seats": 6}, {"class_id": "CL096", "class_date": "2026-11-02", "day_of_week": "월요일", "start_time": "21:00", "end_time": "21:50", "lesson_type": "7:1 레슨", "instructor_name": "노지안", "capacity": 7, "booked_count": 2, "remaining_seats": 5}, {"class_id": "CL021", "class_date": "2026-11-02", "day_of_week": "월요일", "start_time": "21:00", "end_time": "21:50", "lesson_type": "7:1 레슨", "instructor_name": "임채아", "capacity": 7, "booked_count": 1, "remaining_seats": 6}], "message": "예약 가능한 수업 조회가 완료되었습니다."}]
(AI 상담사) 보유 이용권은 7:1 그룹 96회권 잔여 93회(만료 2027-08-31, 사용 가능), 2:1 듀엣 12회권 잔여 12회(만료 2026-10-31, 사용 불가)입니다. 예약 가능 수업: 2026-11-02 18:00 서다은 강사님 7:1 레슨(CL072, 잔여 7석), 2026-11-02 20:00 배수현 강사님 7:1 레슨(CL017, 잔여 5석), 2026-11-02 20:00 임채아 강사님 7:1 레슨(CL032, 잔여 6석), 2026-11-02 21:00 노지안 강사님 7:1 레슨(CL096, 잔여 5석), 2026-11-02 21:00 임채아 강사님 7:1 레슨(CL021, 잔여 6석).
(고객) 그럼 제 예약 내역이랑 취소할 때 횟수 복구 조건을 같이 알려주세요.
(function_call) [{"name": "get_my_bookings", "arguments": {}}, {"name": "search_policy_info", "arguments": {"keyword": "취소"}}]
(function_response) [{"success": true, "user_id": "U001", "user_name": "장현우", "booking_count": 3, "bookings": [{"booking_id": "B0002", "class_id": "CL027", "class_date": "2026-11-06", "day_of_week": "금요일", "start_time": "15:00", "end_time": "15:50", "lesson_type": "7:1 레슨", "instructor_name": "문하린", "booked_at": "2026-10-30 12:01", "status": "CONFIRMED", "week_status": "ACTIVE", "waitlist_position": null, "cancelled_at": null, "user_membership_id": "M001", "credit_deducted": true, "credit_restored": false}, {"booking_id": "B0059", "class_id": "CL098", "class_date": "2026-11-06", "day_of_week": "금요일", "start_time": "21:00", "end_time": "21:50", "lesson_type": "7:1 레슨", "instructor_name": "서다은", "booked_at": "2026-10-30 12:02", "status": "CANCELLED", "week_status": "ACTIVE", "waitlist_position": null, "cancelled_at": "2026-10-30 13:12", "user_membership_id": "M001", "credit_deducted": false, "credit_restored": true}, {"booking_id": "B0001", "class_id": "CL007", "class_date": "2026-11-05", "day_of_week": "목요일", "start_time": "10:00", "end_time": "10:50", "lesson_type": "7:1 레슨", "instructor_name": "임채아", "booked_at": "2026-10-30 12:00", "status": "CONFIRMED", "week_status": "ACTIVE", "waitlist_position": null, "cancelled_at": null, "user_membership_id": "M001", "credit_deducted": true, "credit_restored": false}], "message": "내 예약 내역 조회 완료"}, {"success": true, "policy_count": 4, "policies": [{"policy_id": "P006", "keyword": "취소", "content": "수업 시작 12시간 전까지 취소하면 이용권 1회가 복구됩니다.", "effective_date": "2026-01-01"}, {"policy_id": "P007", "keyword": "취소", "content": "수업 시작 12시간 미만에 취소하면 이용권이 복구되지 않습니다.", "effective_date": "2026-01-01"}, {"policy_id": "P008", "keyword": "취소", "content": "이미 종료된 수업은 직접 취소할 수 없습니다.", "effective_date": "2026-01-01"}, {"policy_id": "P013", "keyword": "환불", "content": "이용권 중도 환불은 센터 상담을 통해 산정하며, 시스템에서 자동 결제 취소를 진행하지 않습니다.", "effective_date": "2026-01-01"}]}]
(AI 상담사) 예약 3건: 2026-11-06 15:00~15:50 문하린 강사님 7:1 레슨(CL027, 예약번호 B0002, CONFIRMED), 2026-11-06 21:00~21:50 서다은 강사님 7:1 레슨(CL098, 예약번호 B0059, CANCELLED), 2026-11-05 10:00~10:50 임채아 강사님 7:1 레슨(CL007, 예약번호 B0001, CONFIRMED). 취소 규정: 수업 시작 12시간 전까지 취소하면 이용권 1회가 복구됩니다. 수업 시작 12시간 미만에 취소하면 이용권이 복구되지 않습니다. 이미 종료된 수업은 직접 취소할 수 없습니다. 이용권 중도 환불은 센터 상담을 통해 산정하며, 시스템에서 자동 결제 취소를 진행하지 않습니다..
(고객) 제 이용권이 없어서 월요일 수업을 못 보는 건 아니죠?
(AI 상담사) 네. 그룹 이용권은 유효하며, 월요일 저녁 수업도 예약 가능한 것으로 조회됐습니다.
(고객) 그룹 이용권은 몇 회 남았어요?
(AI 상담사) 93회 남아 있습니다. 추가로 확인하실 내용이 있을까요?
(고객) 아니요, 이제 없습니다. 감사합니다.
(AI 상담사) 추가로 궁금하신 사항이 없으시면 상담을 종료하겠습니다. 감사합니다!!

**예시 3 | 복합 조건 + 재조회 + FOLLOW_UP (6턴)**
[고객 ID] U001
[대화날짜] 2026-11-02
(고객) 안녕하세요. 제 그룹 이용권으로 월요일이나 수요일 저녁 6시 이후 수업 중 빈자리가 있는지 볼 수 있을까요?
(function_call) [{"name": "get_my_memberships", "arguments": {}}, {"name": "get_available_classes", "arguments": {"day_of_week": "월요일", "lesson_type": "7:1 레슨", "start_time": "18:00", "time_filter": "at_or_after"}}, {"name": "get_available_classes", "arguments": {"day_of_week": "수요일", "lesson_type": "7:1 레슨", "start_time": "18:00", "time_filter": "at_or_after"}}]
(function_response) [{"success": true, "user_id": "U001", "user_name": "장현우", "reference_date": "2026-11-02", "membership_count": 2, "user_memberships": [{"user_membership_id": "M001", "membership_id": "P003", "membership_name": "7:1 그룹 96회권", "product_type": "SINGLE", "lesson_type": "7:1 레슨", "components": [{"lesson_type": "7:1 레슨", "sessions": 96}], "total_sessions": 96, "start_date": "2026-09-01", "expires_at": "2027-08-31", "remaining_sessions": 93, "remaining_sessions_by_type": {"7:1 레슨": 93}, "status": "ACTIVE", "is_usable": true}, {"user_membership_id": "M002", "membership_id": "P004", "membership_name": "2:1 듀엣 12회권", "product_type": "SINGLE", "lesson_type": "2:1 듀엣 레슨", "components": [{"lesson_type": "2:1 듀엣 레슨", "sessions": 12}], "total_sessions": 12, "start_date": "2026-09-01", "expires_at": "2026-10-31", "remaining_sessions": 12, "remaining_sessions_by_type": {"2:1 듀엣 레슨": 12}, "status": "ACTIVE", "is_usable": false}], "message": "내 이용권 조회 완료"}, {"success": true, "week_start_date": "2026-11-02", "available_class_count": 5, "available_classes": [{"class_id": "CL072", "class_date": "2026-11-02", "day_of_week": "월요일", "start_time": "18:00", "end_time": "18:50", "lesson_type": "7:1 레슨", "instructor_name": "서다은", "capacity": 7, "booked_count": 0, "remaining_seats": 7}, {"class_id": "CL017", "class_date": "2026-11-02", "day_of_week": "월요일", "start_time": "20:00", "end_time": "20:50", "lesson_type": "7:1 레슨", "instructor_name": "배수현", "capacity": 7, "booked_count": 2, "remaining_seats": 5}, {"class_id": "CL032", "class_date": "2026-11-02", "day_of_week": "월요일", "start_time": "20:00", "end_time": "20:50", "lesson_type": "7:1 레슨", "instructor_name": "임채아", "capacity": 7, "booked_count": 1, "remaining_seats": 6}, {"class_id": "CL096", "class_date": "2026-11-02", "day_of_week": "월요일", "start_time": "21:00", "end_time": "21:50", "lesson_type": "7:1 레슨", "instructor_name": "노지안", "capacity": 7, "booked_count": 2, "remaining_seats": 5}, {"class_id": "CL021", "class_date": "2026-11-02", "day_of_week": "월요일", "start_time": "21:00", "end_time": "21:50", "lesson_type": "7:1 레슨", "instructor_name": "임채아", "capacity": 7, "booked_count": 1, "remaining_seats": 6}], "message": "예약 가능한 수업 조회가 완료되었습니다."}, {"success": true, "week_start_date": "2026-11-02", "available_class_count": 3, "available_classes": [{"class_id": "CL002", "class_date": "2026-11-04", "day_of_week": "수요일", "start_time": "18:00", "end_time": "18:50", "lesson_type": "7:1 레슨", "instructor_name": "임채아", "capacity": 7, "booked_count": 0, "remaining_seats": 7}, {"class_id": "CL073", "class_date": "2026-11-04", "day_of_week": "수요일", "start_time": "19:00", "end_time": "19:50", "lesson_type": "7:1 레슨", "instructor_name": "임채아", "capacity": 7, "booked_count": 1, "remaining_seats": 6}, {"class_id": "CL078", "class_date": "2026-11-04", "day_of_week": "수요일", "start_time": "20:00", "end_time": "20:50", "lesson_type": "7:1 레슨", "instructor_name": "서다은", "capacity": 7, "booked_count": 0, "remaining_seats": 7}], "message": "예약 가능한 수업 조회가 완료되었습니다."}]
(AI 상담사) 그룹 이용권은 93회 남아 있고 사용 가능합니다. 월요일 오후 6시 서다은 강사님 7:1 수업(CL072)은 7석, 수요일 오후 6시 임채아 강사님 7:1 수업(CL002)은 7석이 남아 있습니다. 다른 저녁 시간대에도 예약 가능한 수업이 있습니다.
(고객) 수요일에 말씀하신 임채아 강사님 수업은 7시에도 있나요? 같은 그룹 레슨으로요.
(function_call) [{"name": "get_available_classes", "arguments": {"day_of_week": "수요일", "lesson_type": "7:1 레슨", "instructor_name": "임채아", "start_time": "19:00", "time_filter": "exact"}}]
(function_response) [{"success": true, "week_start_date": "2026-11-02", "available_class_count": 1, "available_classes": [{"class_id": "CL073", "class_date": "2026-11-04", "day_of_week": "수요일", "start_time": "19:00", "end_time": "19:50", "lesson_type": "7:1 레슨", "instructor_name": "임채아", "capacity": 7, "booked_count": 1, "remaining_seats": 6}], "message": "예약 가능한 수업 조회가 완료되었습니다."}]
(AI 상담사) 네. 수요일 오후 7시~7시 50분 임채아 강사님 7:1 수업(CL073)이 조회되며 잔여 6석입니다.
(고객) 그러면 그 7시 수업도 아까 확인한 이용권으로 들을 수 있는 거죠?
(AI 상담사) 네. 앞에서 확인한 그룹 이용권은 사용 가능하고, 해당 수업도 7:1 레슨이어서 유형이 일치합니다. 다만 예약은 아직 확정하지 않았습니다.
(고객) 수요일 오후 7시 수업은 어떤 강사님인가요?
(AI 상담사) 임채아 강사님의 7:1 그룹 수업입니다.
(고객) 그 수업은 몇 시에 끝나나요?
(AI 상담사) 오후 7시 50분에 끝납니다.
(고객) 좋아요, 궁금한 건 모두 해결됐어요. 감사합니다.
(AI 상담사) 추가로 궁금하신 사항이 없으시면 상담을 종료하겠습니다. 감사합니다!!

**예시 4 | 정말 모순된 조건의 CLARIFY + 해소 후 TOOL_CALL + UNSUPPORTED (6턴)**
[고객 ID] U001
[대화날짜] 2026-11-02
(고객) 7:1 듀엣 수업도 있어요?
(AI 상담사) 7:1 그룹 수업과 2:1 듀엣 수업 중 어느 쪽을 말씀하시는 걸까요?
(고객) 2:1 듀엣 수업이요.
(function_call) [{"name": "get_class_schedule", "arguments": {"lesson_type": "2:1 듀엣 레슨"}}]
(function_response) [{"success": false, "class_count": 0, "class_schedule": [], "message": "조건에 해당하는 시간표가 없습니다."}]
(AI 상담사) 해당 조건으로 조회한 정규 시간표는 없습니다.
(고객) 그럼 월요일 서다은 강사님의 7:1 그룹 시간표 알려주세요.
(function_call) [{"name": "get_class_schedule", "arguments": {"day_of_week": "월요일", "lesson_type": "7:1 레슨", "instructor_name": "서다은"}}]
(function_response) [{"success": true, "class_count": 2, "class_schedule": [{"class_id": "CL043", "day_of_week": "월요일", "start_time": "08:00", "end_time": "08:50", "lesson_type": "7:1 레슨", "instructor_id": "I001", "instructor_name": "서다은", "capacity": 7, "status": "ACTIVE"}, {"class_id": "CL072", "day_of_week": "월요일", "start_time": "18:00", "end_time": "18:50", "lesson_type": "7:1 레슨", "instructor_id": "I001", "instructor_name": "서다은", "capacity": 7, "status": "ACTIVE"}]}]
(AI 상담사) 정규 시간표: 월요일 08:00~08:50 서다은 강사님 7:1 레슨(CL043), 월요일 18:00~18:50 서다은 강사님 7:1 레슨(CL072). 정규 시간표는 예약 가능한 잔여석을 의미하지 않습니다.
(고객) 아, 제가 처음에 7:1 그룹과 2:1 듀엣을 섞어 말해서 물어보신 거군요. 그럼 같은 강사님 수요일에도 있나요?
(function_call) [{"name": "get_class_schedule", "arguments": {"day_of_week": "수요일", "lesson_type": "7:1 레슨", "instructor_name": "서다은"}}]
(function_response) [{"success": true, "class_count": 1, "class_schedule": [{"class_id": "CL078", "day_of_week": "수요일", "start_time": "20:00", "end_time": "20:50", "lesson_type": "7:1 레슨", "instructor_id": "I001", "instructor_name": "서다은", "capacity": 7, "status": "ACTIVE"}]}]
(AI 상담사) 정규 시간표: 수요일 20:00~20:50 서다은 강사님 7:1 레슨(CL078). 정규 시간표는 예약 가능한 잔여석을 의미하지 않습니다.
(고객) 그런데 예약된 수업을 다른 날로 변경해 주실 수 있나요?
(AI 상담사) 현재 제공된 함수는 조회 기능이므로 예약을 직접 변경할 수는 없습니다. 대신 취소 규정을 확인해 드릴 수 있습니다.
(고객) 알겠습니다. 이제 더 궁금한 것은 없어요. 감사합니다!
(AI 상담사) 추가로 궁금하신 사항이 없으시면 상담을 종료하겠습니다. 감사합니다!!
"""
    prompt += f"\n\n[최종 지시] 위 상세 예시를 그대로 복사하지 말고, 지정된 행동 계획에 따라 **새로운 고객 발화 {plan['turn_count']}개와 자연어 최종 답변 {plan['turn_count']}개**만 생성하세요. function_call과 function_response는 요청이 있는 고객 턴 내부에만 넣고 Tool Schema 및 반환 스키마를 준수하세요. 마지막 고객은 질문을 마무리하고, 마지막 상담사 답변은 반드시 '감사합니다!!'로 끝내세요. 출력에는 행동 라벨이나 마크다운 코드블록을 넣지 마세요."
    
    # 전체 대화 계획을 유지한 채 2단계 연속 생성
    # 길이는 유연하게(총 5~6턴), 내용은 엄격하게(JSON/도구 응답/종료)
    # 1단계가 4턴이면 2단계 1~2턴, 3턴이면 2~3턴을 허용한다
    first_count = (plan["turn_count"] + 1) // 2
    tool_schema_by_name = {t["name"]: t for t in tools}

    def _split_turns(raw):
        tags = list(re.finditer(r"(?m)^\(고객\)\s*", raw or ""))
        if not tags:
            return "", []
        prefix = raw[:tags[0].start()].strip()
        turns = [raw[tag.start():tags[i + 1].start() if i + 1 < len(tags) else len(raw)].strip()
                 for i, tag in enumerate(tags)]
        return prefix, turns

    def _valid_chunk(raw, min_needed, max_needed, is_final, action_plan,
                     require_metadata=False):
        """완결된 턴 5~6개를 위한 느슨한 길이 / 엄격한 함수 쌍 검증.

        길이 범위만 유연하게 한다. 각 턴의 JSON, 함수명, 인자, 호출/응답 대응은 유지한다.
        종료 턴은 예정된 인덱스보다 빨리 나와도 허용한다(전체가 5~6턴인 경우만).
        """
        if not raw or not raw.strip():
            return False, "", [], "EMPTY"
        normalized, _ = normalize_generated_dialogue(raw)
        prefix, turns = _split_turns(normalized)
        if not min_needed <= len(turns) <= max_needed:
            return False, prefix, turns, f"TURNS_{len(turns)}_EXPECTED_{min_needed}_TO_{max_needed}"
        if require_metadata and not (re.search(r"\[고객\s*ID\]", prefix) and
                                     re.search(r"\[대화\s*날짜\]", prefix)):
            return False, prefix, turns, "METADATA_MISSING"
        if not require_metadata and re.search(r"\[고객\s*ID\]|\[대화\s*날짜\]", prefix):
            return False, prefix, turns, "DUPLICATED_METADATA"
        for i, turn in enumerate(turns, start=1):
            # 마지막 턴 CLOSE는 예정보다 일찍 도착해도 허용하되 반드시 완결되어야 한다
            action = ("CLOSE" if is_final and i == len(turns)
                      else action_plan[i - 1] if i - 1 < len(action_plan) else None)
            tag_re = re.compile(r"(?m)^\((고객|AI 상담사|function_call|function_response)\)\s*")
            tags = list(tag_re.finditer(turn))
            blocks = [(m.group(1), turn[m.end():tags[j + 1].start()].strip()
                       if j + 1 < len(tags) else turn[m.end():].strip())
                      for j, m in enumerate(tags)]
            kinds = [k for k, _ in blocks]
            if kinds not in (["고객", "AI 상담사"],
                             ["고객", "function_call", "function_response", "AI 상담사"]):
                return False, prefix, turns, f"TURN_{i}_ORDER"
            if any(not val for _, val in blocks):
                return False, prefix, turns, f"TURN_{i}_EMPTY"
            has_tool = "function_call" in kinds
            if action in ("TOOL_CALL", "MULTI_CALL") and not has_tool:
                return False, prefix, turns, f"TURN_{i}_MISSING_TOOL"
            if action == "CLOSE" and has_tool:
                return False, prefix, turns, f"TURN_{i}_CLOSE_HAS_TOOL"
            if has_tool:
                try:
                    calls = json.loads(blocks[1][1])
                    responses = json.loads(blocks[2][1])
                    if isinstance(calls, dict):
                        calls = [calls]
                    if isinstance(responses, dict):
                        responses = [responses]
                except (ValueError, TypeError):
                    return False, prefix, turns, f"TURN_{i}_JSON"
                if (not isinstance(calls, list) or not calls or
                        not isinstance(responses, list) or len(calls) != len(responses)):
                    return False, prefix, turns, f"TURN_{i}_CALL_RESPONSE_COUNT"
                if action == "MULTI_CALL" and len(calls) < 2:
                    return False, prefix, turns, f"TURN_{i}_MULTI_CALL"
                for call, result in zip(calls, responses):
                    if not isinstance(call, dict) or not isinstance(result, dict):
                        return False, prefix, turns, f"TURN_{i}_CALL_TYPE"
                    name, args = call.get("name"), call.get("arguments")
                    if name not in tool_schema_by_name or not isinstance(args, dict):
                        return False, prefix, turns, f"TURN_{i}_FUNCTION"
                    ps = tool_schema_by_name[name].get("parameters", {})
                    props = ps.get("properties", {})
                    if not set(args).issubset(props) or not set(ps.get("required", [])).issubset(args):
                        return False, prefix, turns, f"TURN_{i}_ARGS"
                    for key, value in args.items():
                        if "enum" in props[key] and value not in props[key]["enum"]:
                            return False, prefix, turns, f"TURN_{i}_ENUM"
            if not is_final and re.search(r"감사합니다!!\s*$", blocks[-1][1]):
                return False, prefix, turns, f"TURN_{i}_EARLY_CLOSE"
        if is_final:
            last = turns[-1]
            customer = re.search(r"(?m)^\(고객\)\s*(.*)", last)
            if not customer or not re.search(r"감사|질문.{0,8}없|궁금.{0,8}없|이제 괜찮|상담.{0,8}마", customer.group(1)):
                return False, prefix, turns, "CLOSING_CUSTOMER_MISSING"
            if not last.rstrip().endswith("감사합니다!!"):
                return False, prefix, turns, "CLOSING_ASSISTANT_MISSING"
        return True, prefix, turns, "OK"

    # 출력 예시는 첫 단계에서 충분히 제공. 두 번째 단계에서는 실제 앞선 대화를 반드시 전달
    compact_prompt = prompt.split("[상세 출력 예시]", 1)[0]
    first_stage_prompt = prompt.split("[최종 지시]", 1)[0]
    previous_dialogue = ""
    previous_turn_count = 0
    first_metadata = None

    for part_index in range(2):
        is_final = (part_index == 1)
        if part_index == 0:
            # 전반부는 3턴 권장, 2~4턴까지 수용. 미완성/조기 종료는 거부
            min_needed, max_needed = 2, 4
            preferred = first_count
            actions = plan["actions"][:4]
            segment_prompt = first_stage_prompt
        else:
            # 앞부분이 실제 생성한 길이를 기준으로 뒤쪽 계획을 재계산한다
            # 전체 5~6턴이면 되므로 3+2, 4+1도 성공할 수 있다
            min_needed = max(1, 5 - previous_turn_count)
            max_needed = 6 - previous_turn_count
            preferred = min(max_needed, max(min_needed, plan["turn_count"] - previous_turn_count))
            actions = plan["actions"][previous_turn_count:-1] + ["CLOSE"]
            segment_prompt = compact_prompt

        segment_prompt += (
            f"\n\n[이번 호출의 범위 — 아래 지시를 최우선으로 적용]\n"
            f"전체 고객 5~6턴 중 현재까지 확정된 고객 턴은 {previous_turn_count}개입니다. "
            f"이번에는 이어지는 고객 발화 {preferred}개를 권장하며, "
            f"완결된 {min_needed}~{max_needed}개도 허용됩니다. "
            f"각 고객 발화마다 최종 (AI 상담사) 답변 하나를 작성하세요.\n"
            f"이번 부분의 행동 가이드: {actions}. 행동 라벨은 출력하지 마세요.\n"
            "요구되는 TOOL_CALL/MULTI_CALL은 실제 해당 도구를 호출해야 합니다. "
            "함수 사용 시 (고객) → (function_call) JSON 리스트 → "
            "(function_response) JSON 리스트 → (AI 상담사) 순서. "
            "반환 JSON은 함수별 상세 스키마와 호출 개수를 맞추세요.\n"
        )
        if not is_final:
            segment_prompt += (
                f"[고객 ID] {cid}\n[대화날짜] {CHAT_DATE} 부터 출력하세요. "
                "이번 부분에서는 상담을 종료하거나 감사 종료 인사를 하지 마세요."
            )
        else:
            segment_prompt += (
                "\n[이전 대화 — 참조용. 절대로 재출력하지 마세요]\n"
                + previous_dialogue
                + "\n[이어서 작성] 위의 고객 질문, 함수 호출, 함수 응답, 상담사 답변 전체를 참고하여 "
                "오직 다음 (고객)부터 이어서 작성하세요. 메타데이터도 재출력하지 마세요. "
                "전체 고객 5~6턴이 되도록 이번에는 부족한 수만 생성하면 됩니다. "
                "**이번 부분의 마지막 고객은 더 궁금한 것이 없다고 말하며 감사 인사로 끝내고**, "
                "마지막 (AI 상담사) 답변은 반드시 '감사합니다!!'로 끝내세요. "
                "마지막 턴은 함수 호출이 없는 상담 종료여야 합니다."
            )

        segment_result = None
        last_issue = None
        for retry in range(3):
            stage = part_index + 1
            print(f"[파트 {stage}/2] API 시도 {retry + 1}/3 시작", flush=True)
            try:
                response = client.chat.completions.create(
                    model=MODEL,
                    messages=[
                        {"role": "system", "content": "필라테스 상담 데이터 생성. 현재 요청된 고객 턴만 출력하고, 정확한 함수 호출/응답 및 완결된 최종 답변을 작성하세요."},
                        {"role": "user", "content": segment_prompt},
                    ],
                    temperature=0.35,
                    max_tokens=8500,
                )
                raw = (response.choices[0].message.content or "").strip()
                finish = response.choices[0].finish_reason
                ok, prefix, turns, reason = _valid_chunk(
                    raw, min_needed, max_needed, is_final, actions,
                    require_metadata=(part_index == 0)
                )
                if ok:
                    if part_index == 0:
                        first_metadata = prefix
                        previous_dialogue = first_metadata + "\n" + "\n".join(turns)
                    else:
                        previous_dialogue += "\n" + "\n".join(turns)
                    previous_turn_count += len(turns)
                    segment_result = "\n".join(turns)
                    print(f"[파트 {stage}/2] PASS ({len(turns)}턴, 누적 {previous_turn_count}턴, finish={finish})", flush=True)
                    break
                last_issue = f"검증 오류={reason}, finish={finish}, 응답길이={len(raw)}자"
                print(f"[파트 {stage}/2] FAIL | {last_issue}", flush=True)
                if retry < 2:
                    segment_prompt += (
                        "\n[수정 요청] 이번 부분만 다시 생성하고 미완성 대화/JSON을 출력하지 마세요. "
                        f"고객 턴은 {min_needed}~{max_needed}개로 작성하세요. "
                        f"(오류 코드: {reason}, 종료 사유: {finish})"
                    )
            except openai.AuthenticationError as exc:
                print(f"[파트 {stage}/2] 인증 실패 (HTTP {getattr(exc, 'status_code', 'unknown')}).", flush=True)
                return None
            except openai.OpenAIError as exc:
                last_issue = f"API 오류={type(exc).__name__}, HTTP={getattr(exc, 'status_code', 'N/A')}"
                print(f"[파트 {stage}/2] {last_issue}", flush=True)
        if segment_result is None:
            print(f"[파트 {part_index + 1}/2] 3회 실패 → None. 마지막 원인: {last_issue}", flush=True)
            return None

    # 전체 PASS: 5~6턴, 마지막 CLOSE, 함수/응답 수 일치, 도구 스키마 검증
    final_normalized, _ = normalize_generated_dialogue(previous_dialogue)
    if not quick_check_generated(final_normalized, tools, plan["actions"]):
        reason = check_generated_issues(final_normalized, tools, plan["actions"])
        print(f"[전체 검증] FAIL: {reason[:3]}", flush=True)
        return None
    return final_normalized

# 생성 직후 가벼운 형식 점검. full validation은 Data Transformation에서 수행
def normalize_generated_dialogue(sample):
    """손실 없이 복구 가능한 형식만 표준화: 단일 JSON, 호출 전 단순 안내 멘트.

    고객 5~6턴, 함수 호출 및 반환 JSON 자체는 생성 결과를 유지한다.
    실질적인 답변을 삭제하거나 없는 호출을 만들어내지는 않는다.
    """
    if not isinstance(sample, str) or not sample.strip():
        return sample, []
    tag_re = re.compile(r'^\((고객|AI 상담사|function_call|function_response)\)\s*', re.M)
    matches = list(tag_re.finditer(sample))
    if not matches:
        return sample, []
    prefix = sample[:matches[0].start()].strip()
    blocks = [(m.group(1), sample[m.end():matches[i+1].start()].strip()
               if i+1 < len(matches) else sample[m.end():].strip())
              for i,m in enumerate(matches)]
    repairs = []
    cleaned = []
    i=0
    while i<len(blocks):
        kind, content = blocks[i]
        # customer 직후에 붙은 조회 전 안내멘트: 뒤에 정식 assistant 최종 답변이 있을 때만 제거
        if (kind=='고객' and i+4<len(blocks)
                and [x[0] for x in blocks[i:i+5]] ==
                    ['고객','AI 상담사','function_call','function_response','AI 상담사']):
            intro=blocks[i+1][1]
            if len(intro)<=160 and re.search(r'조회|확인|찾아|살펴|알아보|보겠|보겠습|볼게|안내해 드리',intro):
                cleaned.append(blocks[i])
                repairs.append('호출 전 안내 멘트 제거')
                i+=2
                continue
        if kind in ('function_call','function_response'):
            # JSON 객체 1개를 배열 1개로 표준화(기존 파서의 역할/개수 유지)
            try:
                value=json.loads(content)
                if isinstance(value,dict):
                    content=json.dumps([value],ensure_ascii=False)
                    repairs.append(f'{kind} 단일 객체→리스트')
            except (ValueError,TypeError):
                pass  # 검증 단계에서 FAIL 처리
        cleaned.append((kind,content))
        i+=1
    # 형식 표준화는 데이터 의미에 영향을 주지 않음
    return prefix+'\n'+ '\n'.join(f'({kind}) {content}' for kind,content in cleaned), repairs


def check_generated_issues(sample, tool_schemas, expected_plan=None):
    """PASS/FAIL을 가르는 핵심 무결성 검사만 수행. 행동 라벨 차이는 강제 탈락하지 않음."""
    issues=[]
    if not isinstance(sample,str) or not sample.strip():
        return ['EMPTY']
    sample,_ = normalize_generated_dialogue(sample)
    if not re.search(r'\[고객\s*ID\]\s*\w+',sample):
        issues.append('MISSING_CUSTOMER_ID')
    if not re.search(r'\[대화\s*날짜\]\s*\d{4}-\d{2}-\d{2}',sample):
        issues.append('MISSING_DATE')
    tag_re=re.compile(r'^\((고객|AI 상담사|function_call|function_response)\)\s*',re.M)
    matches=list(tag_re.finditer(sample))
    blocks=[(m.group(1),sample[m.end():matches[i+1].start()].strip()
             if i+1<len(matches) else sample[m.end():].strip())
            for i,m in enumerate(matches)]
    user_count=sum(kind=='고객' for kind,_ in blocks)
    if user_count not in (5, 6):
        issues.append(f'USER_TURNS_{user_count}_NOT_5_OR_6')
    allowed={t['name']:t['parameters'] for t in tool_schemas}
    turns=[]
    current=[]
    for kind,content in blocks:
        if kind=='고객':
            if current: turns.append(current)
            current=[]
        current.append((kind,content))
    if current: turns.append(current)
    tool_turns=0
    for i,turn in enumerate(turns):
        kinds=[k for k,_ in turn]
        if kinds not in (['고객','AI 상담사'],['고객','function_call','function_response','AI 상담사']):
            issues.append(f'TURN_{i+1}_STRUCTURE')
            continue
        if any(not body for _,body in turn):
            issues.append(f'TURN_{i+1}_EMPTY_MESSAGE')
            continue
        has_call='function_call' in kinds
        action=expected_plan[i] if expected_plan and i<len(expected_plan) else None
        if i == len(turns)-1 and re.search(r'감사|질문.{0,8}없|궁금.{0,8}없|다 해결|이제 괜찮', turn[0][1]):
            action = 'CLOSE'  # 5턴으로 조기 종료했더라도 정상 종료 발화를 존중
        # TOOL_CALL/MULTI_CALL 계획에 호출이 없는 것은 FAIL; FOLLOW_UP·CLARIFY의 경계는 엄격 강제하지 않음
        if not has_call:
            if action in ('TOOL_CALL','MULTI_CALL'):
                issues.append(f'TURN_{i+1}_MISSING_TOOL_CALL')
            continue
        tool_turns+=1
        try:
            calls=json.loads(turn[1][1]);responses=json.loads(turn[2][1])
        except (ValueError,TypeError):
            issues.append(f'TURN_{i+1}_INVALID_JSON')
            continue
        if isinstance(calls,dict): calls=[calls]
        if isinstance(responses,dict): responses=[responses]
        if (not isinstance(calls,list) or not calls
                or not isinstance(responses,list) or len(calls)!=len(responses)
                or not all(isinstance(r,dict) for r in responses)):
            issues.append(f'TURN_{i+1}_CALL_RESPONSE_MISMATCH')
            continue
        if action=='MULTI_CALL' and len(calls)<2:
            issues.append(f'TURN_{i+1}_MISSING_MULTI_CALL')
        for call in calls:
            if not isinstance(call,dict):
                issues.append(f'TURN_{i+1}_INVALID_CALL');continue
            name,args=call.get('name'),call.get('arguments')
            if name not in allowed or not isinstance(args,dict):
                issues.append(f'TURN_{i+1}_INVALID_FUNCTION');continue
            ps=allowed[name];props=ps.get('properties',{})
            if not set(args).issubset(props) or not set(ps.get('required',[])).issubset(args):
                issues.append(f'TURN_{i+1}_INVALID_ARGUMENTS');continue
            for key,value in args.items():
                param=props[key]
                if 'enum' in param and value not in param['enum']:
                    issues.append(f'TURN_{i+1}_INVALID_ENUM')
    if tool_turns == 0:
        issues.append('NO_TOOL_CALLS')
    return issues


def quick_check_generated(sample, tool_schemas, expected_plan=None):
    return not check_generated_issues(sample, tool_schemas, expected_plan)


In [48]:
# 3건만 별도 테스트(5~6턴): PASS/FAIL 및 핵심 원인 한 줄만 확인(전체 JSON 출력 없음)
cids = [f"U{i:03d}" for i in range(1, 11)]
sample_results = []
for idx in tqdm(range(3), desc="샘플 3건 테스트"):
    raw = make_function_calling_data(
        tools, cids, tools_return_format, question_topics, unsupported_scenarios,
        plan=make_generation_plan(idx)
    )
    candidate, repairs = normalize_generated_dialogue(raw)
    sample_results.append(candidate)
    issues = check_generated_issues(candidate, tools, make_generation_plan(idx)["actions"])
    status = "PASS" if not issues else "FAIL"
    detail = f" | {issues[0]}" if issues else ""
    print(f"[{idx+1}/3] {status}{detail}" + (f" | 자동정리 {len(repairs)}건" if repairs else ""))


샘플 3건 테스트:   0%|          | 0/3 [00:00<?, ?it/s]

[파트 1/2] API 시도 1/3 시작
[파트 1/2] API 오류=RateLimitError, HTTP=429
[파트 1/2] API 시도 2/3 시작
[파트 1/2] API 오류=RateLimitError, HTTP=429
[파트 1/2] API 시도 3/3 시작
[파트 1/2] API 오류=RateLimitError, HTTP=429
[파트 1/2] 3회 실패 → None. 마지막 원인: API 오류=RateLimitError, HTTP=429
[1/3] FAIL | EMPTY
[파트 1/2] API 시도 1/3 시작
[파트 1/2] API 오류=RateLimitError, HTTP=429
[파트 1/2] API 시도 2/3 시작
[파트 1/2] API 오류=RateLimitError, HTTP=429
[파트 1/2] API 시도 3/3 시작
[파트 1/2] API 오류=RateLimitError, HTTP=429
[파트 1/2] 3회 실패 → None. 마지막 원인: API 오류=RateLimitError, HTTP=429
[2/3] FAIL | EMPTY
[파트 1/2] API 시도 1/3 시작
[파트 1/2] PASS (4턴, 누적 4턴, finish=stop)
[파트 2/2] API 시도 1/3 시작
[파트 2/2] API 오류=RateLimitError, HTTP=429
[파트 2/2] API 시도 2/3 시작
[파트 2/2] PASS (1턴, 누적 5턴, finish=stop)
[3/3] PASS


In [49]:

import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

# 설정
MAX_WORKERS = 2
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

# 기존 PASS 체크포인트 불러오기
if CHECKPOINT_PATH.exists():
    with open(CHECKPOINT_PATH, encoding="utf-8") as f:
        saved = json.load(f)

    result_lst = []

    for idx, raw in enumerate(saved):
        candidate, _ = normalize_generated_dialogue(raw)

        if quick_check_generated(
            candidate, tools, make_generation_plan(idx)["actions"]
        ):
            result_lst.append(candidate)
else:
    result_lst = []

initial_pass = len(result_lst)
attempts = 0
rejected = 0
start_time = time.monotonic()

max_attempts = max(
    30, (NUM_SAMPLES - initial_pass) * 5
)

print(f"체크포인트: {CHECKPOINT_PATH.resolve()}")
print(f"목표: {NUM_SAMPLES}건 | 기존 PASS: {initial_pass}건")
print(f"동시 생성: {MAX_WORKERS}개")

# 초록색 진행 바
pbar = tqdm(
    total=NUM_SAMPLES,
    initial=min(initial_pass, NUM_SAMPLES),
    desc="PASS 대화 수집",
    unit="건",
    colour="green",
    bar_format=(
        "{desc}: {percentage:3.0f}%|{bar}| "
        "{n_fmt}/{total_fmt} "
        "[경과 {elapsed}, 남은시간 {remaining}{postfix}]"
    )
)

def update_progress():
    elapsed = time.monotonic() - start_time
    new_pass = len(result_lst) - initial_pass

    if new_pass > 0:
        seconds_per_pass = elapsed / new_pass
        remaining = NUM_SAMPLES - len(result_lst)
        eta = f"{remaining * seconds_per_pass / 60:.1f}분"
    else:
        eta = "계산 중"

    pbar.set_postfix({
        "PASS": len(result_lst),
        "FAIL": rejected,
        "시도": attempts,
        "경과": f"{elapsed / 60:.1f}분",
        "예상": eta
    }, refresh=True)

def save_checkpoint():
    if not result_lst:
        return

    temp_path = CHECKPOINT_PATH.with_suffix(".tmp")

    with open(temp_path, "w", encoding="utf-8") as f:
        json.dump(
            result_lst,
            f,
            ensure_ascii=False,
            indent=2
        )

    temp_path.replace(CHECKPOINT_PATH)

# 대화 1개 생성
def generate_one(idx, plan):
    result = make_function_calling_data(
        tools,
        cids,
        tools_return_format,
        question_topics,
        unsupported_scenarios,
        plan=plan
    )

    candidate, _ = normalize_generated_dialogue(result)

    is_valid = quick_check_generated(
        candidate,
        tools,
        plan["actions"]
    )

    return candidate if is_valid else None

# 인덱스마다 고정된 행동 계획
plans = {}

# 완료 순서가 달라도 체크포인트는 원래 순서 유지
pending = {}

update_progress()

try:
    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:

        while (
            len(result_lst) < NUM_SAMPLES
            and attempts < max_attempts
        ):
            remaining_attempts = max_attempts - attempts

            # 최대 2개 서로 다른 대화 동시 생성
            indices = [
                i
                for i in range(
                    len(result_lst),
                    min(
                        len(result_lst) + MAX_WORKERS,
                        NUM_SAMPLES
                    )
                )
                if i not in pending
            ][:remaining_attempts]

            if not indices:
                break

            futures = {}

            for idx in indices:
                if idx not in plans:
                    plans[idx] = make_generation_plan(idx)

                future = executor.submit(
                    generate_one,
                    idx,
                    plans[idx]
                )

                futures[future] = idx

            # 완료되는 순서대로 결과 수집
            for future in as_completed(futures):
                idx = futures[future]
                attempts += 1

                try:
                    candidate = future.result()
                except Exception as e:
                    candidate = None
                    tqdm.write(
                        f"[{idx}] 오류: {type(e).__name__}"
                    )

                if candidate is None:
                    rejected += 1
                else:
                    pending[idx] = candidate

                # 앞 번호부터 순서대로 PASS 확정
                previous_count = len(result_lst)

                while len(result_lst) in pending:
                    next_idx = len(result_lst)
                    result_lst.append(pending.pop(next_idx))
                    pbar.update(1)

                # 체크포인트 저장
                if len(result_lst) > previous_count:
                    crossed_save_point = (
                        len(result_lst) // SAVE_EVERY
                        > previous_count // SAVE_EVERY
                    )

                    if (
                        crossed_save_point
                        or len(result_lst) == NUM_SAMPLES
                    ):
                        save_checkpoint()

                update_progress()

finally:
    save_checkpoint()
    update_progress()
    pbar.close()

# 결과
total_minutes = (
    time.monotonic() - start_time
) / 60

print(f"본 생성 PASS: {len(result_lst)}/{NUM_SAMPLES}")
print(f"FAIL: {rejected}건 | 총 시도: {attempts}회")
print(f"총 경과 시간: {total_minutes:.1f}분")
print(f"체크포인트: {CHECKPOINT_PATH.resolve()}")

if len(result_lst) < NUM_SAMPLES:
    raise RuntimeError(
        "목표 PASS 개수를 채우지 못했습니다. "
        "저장된 체크포인트부터 다시 실행하세요."
    )


체크포인트: /workspace/outputs_v2/generation_checkpoint_5to6_close_120.json
목표: 80건 | 기존 PASS: 82건
동시 생성: 2개


PASS 대화 수집: 100%|##########| 80/80 [경과 00:00, 남은시간 ?]

본 생성 PASS: 82/80
FAIL: 0건 | 총 시도: 0회
총 경과 시간: 0.0분
체크포인트: /workspace/outputs_v2/generation_checkpoint_5to6_close_120.json


In [50]:
import os
import glob

print("현재 작업 경로:", os.getcwd())

files = glob.glob("**/*checkpoint*.json", recursive=True)

print("발견된 체크포인트:", files)

현재 작업 경로: /workspace
발견된 체크포인트: ['outputs_v2/generation_checkpoint_5to6_close_120.json']


## **7. Data Transformation**

In [51]:
# 1. 고객 아이디와 대화 날짜 추출
def parse_metadata(text: str):
    customer_id_match = re.search(r'\[고객 ID\]\s*(\w+)', text)
    date_match = re.search(r'\[대화\s*날짜\]\s*(\d{4}-\d{2}-\d{2})', text)

    customer_id = customer_id_match.group(1) if customer_id_match else None
    date = date_match.group(1) if date_match else None

    return customer_id, date


# 2. 원본 대화를 Qwen ChatML 호환 messages로 변환
def parse_to_qwen_format(text: str):
    messages = []

    pattern = re.compile(
        r'^\((고객|AI 상담사|function_call|function_response)\)\s*'
        r'(.*?)(?=^\((?:고객|AI 상담사|function_call|function_response)\)|\Z)',
        re.MULTILINE | re.DOTALL
    )

    pending_calls = 0

    for match in pattern.finditer(text.strip()):
        role_type, content = match.groups()
        content = content.strip()

        if role_type == "고객":
            if pending_calls:
                raise ValueError("함수 반환 결과가 누락되었습니다.")

            messages.append({
                "role": "user",
                "content": content
            })

        elif role_type == "function_call":
            calls = json.loads(content)

            # 단일 JSON 객체도 허용하되 최종 학습 데이터에서는 리스트로 표준화
            if isinstance(calls, dict):
                calls = [calls]
            if not isinstance(calls, list) or not calls:
                raise ValueError("function_call은 비어 있지 않은 JSON 리스트/객체여야 합니다.")

            # 여러 함수 호출을 하나의 assistant 메시지에 묶음
            tool_call_text = "\n".join(
                f"<tool_call>\n"
                f"{json.dumps(call, ensure_ascii=False)}\n"
                f"</tool_call>"
                for call in calls
            )

            messages.append({
                "role": "assistant",
                "content": tool_call_text
            })

            pending_calls += len(calls)

        elif role_type == "function_response":
            responses = json.loads(content)

            if not isinstance(responses, list):
                responses = [responses]

            if len(responses) != pending_calls:
                raise ValueError(
                    f"함수 호출 {pending_calls}개, "
                    f"반환 결과 {len(responses)}개로 개수가 다릅니다."
                )

            # 도구 결과는 assistant 학습 정답이 아님
            for response in responses:
                messages.append({
                    "role": "tool",
                    "content": json.dumps(response, ensure_ascii=False)
                })

            pending_calls = 0

        elif role_type == "AI 상담사":
            if pending_calls:
                raise ValueError("함수 결과를 받기 전에 상담사가 답변했습니다.")

            messages.append({
                "role": "assistant",
                "content": content
            })

    if pending_calls:
        raise ValueError("마지막 함수 호출의 반환 결과가 누락되었습니다.")

    return messages

# 파싱 예시
example = next((x for x in result_lst if isinstance(x, str)), None)
if example:
    cid, date = parse_metadata(example)
    print("cid:", cid, "date:", date)
    print(parse_to_qwen_format(example)[:4])


cid: U006 date: 2026-11-02
[{'role': 'user', 'content': '안녕하세요, 필라테스 수업 시간표 좀 알려주세요.'}, {'role': 'assistant', 'content': '<tool_call>\n{"name": "get_class_schedule", "arguments": {}}\n</tool_call>'}, {'role': 'tool', 'content': '{"success": true, "class_count": 5, "class_schedule": [{"class_id": "CL001", "day_of_week": "월요일", "start_time": "09:00", "end_time": "09:50", "lesson_type": "7:1 레슨", "instructor_id": "I002", "instructor_name": "김민지", "capacity": 7, "status": "ACTIVE"}, {"class_id": "CL002", "day_of_week": "화요일", "start_time": "10:00", "end_time": "10:50", "lesson_type": "2:1 듀엣 레슨", "instructor_id": "I003", "instructor_name": "박지훈", "capacity": 2, "status": "ACTIVE"}, {"class_id": "CL003", "day_of_week": "수요일", "start_time": "11:00", "end_time": "11:50", "lesson_type": "1:1 개인 레슨", "instructor_id": "I004", "instructor_name": "이수진", "capacity": 1, "status": "ACTIVE"}, {"class_id": "CL004", "day_of_week": "목요일", "start_time": "12:00", "end_time": "12:50", "lesson_type": "7:1 레슨

In [53]:
# 기존 parser 결과 검증 강화. GPT 재호출 없이 실제 DB 결과로 Tool Response만 교체
GROUND_WITH_DB = True
MAX_TOOL_ITEMS = 5

TOOL_HANDLERS = {
    "get_membership_info": get_membership_info,
    "get_class_schedule": get_class_schedule,
    "get_available_classes": get_available_classes,
    "get_my_memberships": get_my_memberships,
    "get_my_bookings": get_my_bookings,
    "search_policy_info": search_policy_info
}
TOOL_CALL_PATTERN = re.compile(r'<tool_call>\s*(.*?)\s*</tool_call>', re.S)
SCHEMA_BY_NAME = {x['name']: x['parameters'] for x in tools}

def compact_tool_json(obj, limit=MAX_TOOL_ITEMS):
    """개수/상태값은 보존하고, 긴 list는 앞쪽 일부와 전체 건수를 함께 제공."""
    if isinstance(obj, dict):
        return {k: compact_tool_json(v, limit) for k, v in obj.items()}
    if isinstance(obj, list):
        if len(obj) > limit:
            return {
                "items": [compact_tool_json(v, limit) for v in obj[:limit]],
                "total_items": len(obj),
                "truncated": True
            }
        return [compact_tool_json(v, limit) for v in obj]
    return obj

def validate_dialogue(messages):
    """고객 5~6발화, tool->최종 assistant, args/enum 및 필수 응답 검증."""
    if not messages or messages[0]['role'] != 'user':
        raise ValueError('첫 발화 user 누락')

    idx = 0
    user_count = 0
    final_count = 0
    tool_count = 0
    direct_count = 0

    while idx < len(messages):
        if messages[idx]['role'] != 'user':
            raise ValueError(f'{idx}: user 발화 순서 오류')

        user_count += 1
        idx += 1

        if idx >= len(messages) or messages[idx]['role'] != 'assistant':
            raise ValueError('assistant 누락')

        content = messages[idx]['content'].strip()
        calls = TOOL_CALL_PATTERN.findall(content)

        if calls:
            tool_count += 1

            if TOOL_CALL_PATTERN.sub('', content).strip():
                raise ValueError('호출 메시지에 자연어 안내 멘트가 섞임')

            for raw in calls:
                call = json.loads(raw)
                name, arguments = call.get('name'), call.get('arguments')

                if name not in TOOL_HANDLERS or not isinstance(arguments, dict):
                    raise ValueError(f'함수 또는 arguments 오류: {name}')

                ps = SCHEMA_BY_NAME[name]

                if not set(arguments).issubset(ps.get('properties', {})):
                    raise ValueError(f'{name}: 스키마에 없는 인자')

                if not set(ps.get('required', [])).issubset(arguments):
                    raise ValueError(f'{name}: required 인자 부족')

                for key, value in arguments.items():
                    p = ps['properties'][key]
                    if 'enum' in p and value not in p['enum']:
                        raise ValueError(f'{name}.{key}: enum 오류 {value}')

            idx += 1

            for _ in calls:
                if idx >= len(messages) or messages[idx]['role'] != 'tool':
                    raise ValueError('함수 호출과 tool 응답 개수 불일치')
                idx += 1

            if idx >= len(messages) or messages[idx]['role'] != 'assistant':
                raise ValueError('tool 응답 뒤 자연어 최종 답변 없음')

            if TOOL_CALL_PATTERN.search(messages[idx]['content']):
                raise ValueError('tool 응답 뒤 중첩 호출')

        else:
            direct_count += 1

        final_count += 1
        idx += 1

    if user_count not in (5, 6) or final_count != user_count:
        raise ValueError(
            f'고객/최종답변 5~6턴 불일치: {user_count}/{final_count}'
        )

    if tool_count == 0:
        raise ValueError('함수 호출이 전혀 없음')

    return {
        "turns": user_count,
        "tool_turns": tool_count,
        "no_tool_turns": direct_count
    }

def replay_tool_results(messages, cid, date):
    """기존 Function Call과 AI 답변은 유지하고, Tool Response만 실제 DB 결과로 교체."""
    global CURRENT_USER_ID, REFERENCE_DATE

    original_cid, original_date = CURRENT_USER_ID, REFERENCE_DATE
    CURRENT_USER_ID, REFERENCE_DATE = cid, f'{date} 13:00'

    rebuilt = []
    idx = 0

    try:
        while idx < len(messages):
            msg = messages[idx]
            calls = (
                TOOL_CALL_PATTERN.findall(msg['content'])
                if msg['role'] == 'assistant'
                else []
            )

            if not calls:
                rebuilt.append(msg)
                idx += 1
                continue

            parsed = [json.loads(x) for x in calls]
            rebuilt.append(msg)

            for call in parsed:
                result = TOOL_HANDLERS[call['name']](**call['arguments'])

                short = compact_tool_json(
                    json.loads(
                        json.dumps(result, ensure_ascii=False, default=str)
                    )
                )

                rebuilt.append({
                    'role': 'tool',
                    'content': json.dumps(short, ensure_ascii=False)
                })

            end = idx + 1 + len(parsed)

            if end >= len(messages) or messages[end]['role'] != 'assistant':
                raise ValueError('함수 실행 후 원본 assistant 응답 누락')

            # GPT 재호출 없이 기존 AI 상담사 답변 유지
            rebuilt.append(messages[end])
            idx = end + 1

        return rebuilt

    finally:
        CURRENT_USER_ID, REFERENCE_DATE = original_cid, original_date

# 기존에 수집한 데이터만 검증하고 DB 응답 교체. 추가 GPT 생성 없음
records = []
parse_errors = []

def process_final_pass(sample, sample_idx):
    if not isinstance(sample, str) or not sample.strip():
        raise ValueError('빈 응답')

    cid, date = parse_metadata(sample)

    if not cid or not date:
        raise ValueError('고객 ID/날짜 누락')

    msgs = parse_to_qwen_format(sample)
    stats = validate_dialogue(msgs)

    if GROUND_WITH_DB:
        msgs = replay_tool_results(msgs, cid, date)
        stats = validate_dialogue(msgs)

    return {
        'sample_id': f'v2_{sample_idx:04d}',
        'tools': tools,
        'cid': cid,
        'dates': date,
        'messages': msgs,
        **stats
    }

# 기존 PASS 후보만 검증 (새로운 데이터 생성 없음)
for idx, sample in enumerate(tqdm(result_lst, desc='DB 검증 PASS 수집')):
    if len(records) >= NUM_SAMPLES:
        break

    try:
        records.append(process_final_pass(sample, idx))
    except Exception as exc:
        parse_errors.append((idx, str(exc)[:200]))

df = pd.DataFrame(records)

print(f'검증 성공: {len(df)} / 실패: {len(parse_errors)}')
print(f'목표 데이터 수: {NUM_SAMPLES}')

if not df.empty:
    print('평균 Tool Call 턴/대화:', round(df['tool_turns'].mean(), 2))
    print('평균 비호출 턴/대화:', round(df['no_tool_turns'].mean(), 2))
    print('총 Assistant 최종 판단 턴:', int(df['turns'].sum()))

if parse_errors:
    print(
        f'최종 변환 FAIL: {len(parse_errors)}건 (앞 3건 원인):',
        parse_errors[:3]
    )

if len(df) != NUM_SAMPLES:
    print(
        f'주의: 목표 {NUM_SAMPLES}개 중 {len(df)}개만 통과했습니다. '
        '추가 API 호출이나 데이터 생성은 수행하지 않습니다.'
    )

display(df.head())

# CLARIFY 판단의 의미적 정확성은 정규식만으로 확정할 수 없음
# 검증 통과 샘플의 실제 사용자 질문과 추가 질문 12건을 수동 점검
valid_sample_ids = set(df["sample_id"]) if not df.empty else set()
clarify_review = []

for original_idx, original in enumerate(result_lst):
    if f"v2_{original_idx:04d}" not in valid_sample_ids:
        continue

    plan_actions = make_generation_plan(original_idx)['actions']

    if 'CLARIFY' not in plan_actions:
        continue

    segments = [
        p for p in re.split(r'(?=^\(고객\)\s*)', original, flags=re.M)
        if p.startswith('(고객)')
    ]

    for turn_idx, (segment, planned_action) in enumerate(
        zip(segments, plan_actions), start=1
    ):
        if planned_action != 'CLARIFY':
            continue

        customer = re.search(r'^\(고객\)\s*(.+)$', segment, re.M)
        assistant = re.search(r'^\(AI 상담사\)\s*(.+)$', segment, re.M)

        clarify_review.append({
            'sample_id': f'v2_{original_idx:04d}',
            'turn': turn_idx,
            '회원 질문': customer.group(1).strip() if customer else '',
            '모델이 학습할 추가 질문': assistant.group(1).strip() if assistant else '',
        })

if clarify_review:
    print(
        'CLARIFY 검수 대상:',
        len(clarify_review),
        '턴 (질문 필요성이 타당한지 직접 확인)'
    )

    display(
        pd.DataFrame(clarify_review).sample(
            n=min(12, len(clarify_review)),
            random_state=42
        )
    )

DB 검증 PASS 수집:   0%|          | 0/82 [00:00<?, ?it/s]

검증 성공: 80 / 실패: 0
목표 데이터 수: 80
평균 Tool Call 턴/대화: 2.92
평균 비호출 턴/대화: 2.22
총 Assistant 최종 판단 턴: 412


,sample_id,tools,cid,dates,messages,turns,tool_turns,no_tool_turns
0,v2_0000,"[{'type': 'function', 'name': 'get_membership_...",U006,2026-11-02,"[{'role': 'user', 'content': '안녕하세요, 필라테스 수업 시...",6,2,4
1,v2_0001,"[{'type': 'function', 'name': 'get_membership_...",U009,2026-11-02,"[{'role': 'user', 'content': '안녕하세요.'}, {'role...",5,2,3
2,v2_0002,"[{'type': 'function', 'name': 'get_membership_...",U009,2026-11-02,"[{'role': 'user', 'content': '안녕하세요. 제 듀엣 이용권으...",6,3,3
3,v2_0003,"[{'type': 'function', 'name': 'get_membership_...",U003,2026-11-02,"[{'role': 'user', 'content': '안녕하세요. 제 이용권의 시작...",5,3,2
4,v2_0004,"[{'type': 'function', 'name': 'get_membership_...",U009,2026-11-02,"[{'role': 'user', 'content': '7:1 듀엣 수업도 있어요?'...",5,3,2


CLARIFY 검수 대상: 30 턴 (질문 필요성이 타당한지 직접 확인)


,sample_id,turn,회원 질문,모델이 학습할 추가 질문
27,v2_0072,3,7:1과 듀엣 중 어느 수업을 말씀하시는 건가요?,7:1 레슨과 2:1 듀엣 레슨 중 어느 수업을 말씀하시는 건가요? 각각의 수업에 ...
15,v2_0040,3,월요일이나 수요일 중에 가능한 수업이 있는지 알려주세요.,"월요일에는 10:00 박수진 강사님, 14:00 김민수 강사님, 18:00 이서연 ..."
23,v2_0060,1,안녕하세요. 제 이용권으로 가능한 수업이 언제인지 알고 싶어요.,안녕하세요! 이용권에 따라 가능한 수업을 확인해 드리겠습니다. 혹시 특정 요일이나 ...
17,v2_0044,1,7:1 듀엣 수업도 있어요?,7:1 그룹 수업과 2:1 듀엣 수업 중 어느 쪽을 말씀하시는 걸까요?
8,v2_0020,1,7:1 듀엣 수업도 있어요?,7:1 그룹 수업과 2:1 듀엣 수업 중 어느 쪽을 말씀하시는 걸까요?
9,v2_0024,3,1:1 개인 레슨과 2:1 듀엣 레슨의 차이가 뭐예요?,1:1 개인 레슨은 강사와 회원이 1:1로 수업을 진행하여 개인 맞춤형 지도가 가능...
28,v2_0074,2,그럼 월요일에 1:1 개인 레슨 시간표를 보여주세요.,월요일 1:1 개인 레슨 시간표는 다음과 같습니다: 오전 9시부터 9시 50분까지 ...
24,v2_0064,3,7:1 듀엣 수업도 있나요?,7:1 그룹 수업과 2:1 듀엣 수업 중 어느 쪽을 말씀하시는 걸까요?
12,v2_0032,3,7:1 듀엣 수업도 있나요?,7:1 그룹 수업과 2:1 듀엣 수업 중 어느 쪽을 말씀하시는 걸까요?
0,v2_0000,3,7:1 듀엣 수업도 있나요?,7:1 그룹 수업과 2:1 듀엣 수업 중 어느 쪽을 말씀하시는 걸까요?


## **8. System Prompt**

In [ ]:
# Qwen Function Calling용 System Prompt 생성
def generate_qwen_system_prompt(
    tools: list,
    cid: str,
    current_date: str = "2026-11-02"
) -> str:

    # 1. 시스템 프롬프트 기본 정보
    header = """당신은 글로리 필라테스의 AI 상담사입니다. 정확하고 자연스럽게 상담하십시오.

행동 선택 규칙:
- 이미 알고 있는 정보로 답할 수 있으면 바로 답하고, 실제 센터/회원 DB 정보가 필요하면 Tool을 사용합니다.
- 선택 인자가 누락되어도 전체 조회가 가능하면 강사·요일 등을 불필요하게 되묻지 않고 바로 Tool을 호출합니다.
- 모호한 지시 대상이나 서로 충돌하는 조건처럼 Tool·대화 맥락으로도 해소할 수 없는 필수 정보가 있을 때만 한 가지를 되묻습니다.
- 사용자가 그 모호함을 풀어주면 다시 물어보지 말고 곧바로 필요한 작업을 처리합니다.
- 월요일·수요일처럼 양립 가능한 복수 조건은 필요 시 여러 Tool을 호출합니다.
- 이전 Tool 결과만으로 답할 수 있으면 불필요하게 재조회하지 않습니다.
- 예약 생성·취소·결제 등 Tool에 없는 작업은 실행했다고 말하지 않습니다.
- 함수 호출 전 별도의 자연어 안내 멘트를 만들지 않습니다.

로그인한 사용자의 현재 ID: %s
오늘 날짜: %s

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
""" % (cid, current_date)

    # 2. 함수 목록 순서 무작위 변경
    shuffled_tools = tools[:]
    random.shuffle(shuffled_tools)

    # 3. 사용 가능한 함수 정의 추가
    tool_defs = ""

    for tool in shuffled_tools:
        entry = {
            "type": "function",
            "function": {
                "name": tool["name"],
                "description": tool["description"],
                "parameters": tool["parameters"]
            }
        }

        tool_defs += json.dumps(entry, ensure_ascii=False) + "\n"

    # 4. Function Calling 출력 형식
    footer = """</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call>"""

    return header + tool_defs + footer


# 고객별 시스템 프롬프트 생성
if not df.empty:
    df["system_prompt"] = [
        generate_qwen_system_prompt(tool_list, cid, today_date)
        for tool_list, cid, today_date in zip(df["tools"], df["cid"], df["dates"])
    ]
    print(df["system_prompt"].iloc[min(5, len(df) - 1)])
else:
    df["system_prompt"] = pd.Series(dtype="object")


## **9. Save**

In [ ]:
# [V2] 학습/테스트를 대화 단위로 딱 한 번 나눠 Hugging Face에 업로드
# 기존 로컬 JSONL 보관; 별도 V2 리포지토리를 사용하여 옛 학습과 혼동 방지
if df.empty:
    raise ValueError("저장할 데이터가 없음")
output_jsonl = OUTPUT_DIR / "pilates_qwen_training_data_v2.jsonl"
with open(output_jsonl,"w",encoding="utf-8") as f:
    for row in df.to_dict("records"):
        f.write(json.dumps(row,ensure_ascii=False,default=str)+"\n")
print("로컬 저장:", output_jsonl)

from datasets import DatasetDict
all_data=datasets.Dataset.from_pandas(df,preserve_index=False)
splits=all_data.train_test_split(test_size=0.2,seed=42,shuffle=True)
final_dataset=DatasetDict({"train":splits["train"],"test":splits["test"]})
print("Train:",len(final_dataset['train']),"Test:",len(final_dataset['test']))
print("겹친 sample_id:",len(set(final_dataset['train']['sample_id']) & set(final_dataset['test']['sample_id'])))

# Setting 셀에서 검증한 Hugging Face 계정으로 업로드
final_dataset.push_to_hub(HF_REPO_ID, private=True, token=HF_TOKEN)
print("V2 업로드 완료:", HF_REPO_ID)
